# Notebook 03 — Hospital imagery and experimental AI training

[Open the final map (Notebook 01)](https://colab.research.google.com/drive/1YYpGBo-imloznVD9yRzQ4zDmC7EwVOuJ)

This is research/training; it is **not** necessary for regular map use.


## 1 — Install packages
Run this once per runtime. The training library installs here; no model is trained until reviewed labels pass the checks.

In [ ]:
%pip -q install "ultralytics>=8.3,<9" "rasterio>=1.4,<2" "pyproj>=3.4,<4" "Pillow>=10,<13" "pandas>=2,<3" "requests>=2,<3" "shapely>=2,<3" "pyyaml>=6,<7" "overturemaps>=2.1,<3"


## 2 — Persistent project folder and source code
After approving the Google Drive mount, real imagery, annotations, checkpoint weights, evaluation and prediction GeoJSON stay in your Drive folder (`MyDrive/OSS4008_Hospital_AI_Pilot`). The notebook writes two small verified Python modules into the Colab runtime. These are also stored separately in the project's `02_App` folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/OSS4008_Hospital_AI_Pilot')
ROOT.mkdir(parents=True, exist_ok=True)
print('AI pilot workspace:',ROOT)


In [ ]:
from pathlib import Path
import importlib.util, sys
HOSPITAL_AI_SOURCE = '"""Hospital Infrastructure Canada: reproducible geospatial hospital-building detector.\n\nThis module never treats ODHF coordinates, OSM geometries, or unlabeled imagery as\ntraining ground truth. It generates candidate image chips, requires human-reviewed\nLabelMe annotations, enforces site-disjoint YOLO splits, trains/evaluates an actual\nmodel, and exports pixel predictions in the existing map\'s GeoJSON schema.\n\nThe main Colab dashboard is NOT imported or modified by this module.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nfrom io import BytesIO\nimport json\nimport math\nfrom pathlib import Path\nimport random\nimport shutil\nfrom typing import Any\nfrom urllib.parse import urlparse\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageOps, ImageDraw\nimport requests\nimport rasterio\nfrom rasterio.transform import from_bounds\nfrom rasterio.warp import transform as transform_coordinates\nfrom pyproj import Transformer\nimport yaml\n\n# Published Geospatial Ontario imagery service. The relevant DRAPE 2024 imagery\n# is described under the Open Government Licence - Ontario. Check coverage and\n# service metadata for every downloaded site and preserve source attribution.\nONTARIO_IMAGERY_SERVICE = (\n    "https://ws.geoservices.lrc.gov.on.ca/arcgis5/rest/services/"\n    "AerialImagery/GEO_Imagery_Data_Service_2023to2027/ImageServer"\n)\nIMAGERY_LICENSE_URL = "https://data.ontario.ca/dataset/open-ontario-imagery"\nCOLUMN_NAMES = [\n    "tile_id", "site_id", "name", "candidate_class", "latitude", "longitude",\n    "image", "geotiff", "annotation", "imagery_source", "imagery_license_url",\n    "source_extent_3857", "image_width", "image_height", "ground_width_m",\n    "review_status", "reviewed_label", "reviewer", "source_checked",\n]\n\n\ndef sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef _site_token(row: pd.Series) -> str:\n    text = f"{row.get(\'id\', \'\')}|{row.get(\'name\', \'\')}|{float(row.latitude):.6f}|{float(row.longitude):.6f}"\n    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]\n\n\ndef _great_circle_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:\n    r = math.pi / 180\n    a = math.sin((lat2 - lat1) * r / 2) ** 2 + math.cos(lat1 * r) * math.cos(lat2 * r) * math.sin((lon2 - lon1) * r / 2) ** 2\n    return 6371.0088 * 2 * math.asin(min(1, math.sqrt(a)))\n\n\ndef select_review_sites(\n    odhf_csv: str | Path,\n    center_lat: float = 45.4215,\n    center_lon: float = -75.6972,\n    radius_km: float = 30.0,\n    hospitals: int = 12,\n    controls: int = 12,\n    min_site_spacing_m: float = 750,\n) -> pd.DataFrame:\n    """Sample known sites and *unverified* comparison facilities for imagery review.\n\n    `controls` are comparison *candidates*, NOT verified nonhospital buildings.\n    Every candidate requires independent visual annotation before training.\n    """\n    if not (1 <= radius_km <= 150 and 1 <= hospitals <= 120 and 0 <= controls <= 60):\n        raise ValueError("Radius must be 1..150 km, hospitals 1..120, controls 0..60")\n    df = pd.read_csv(odhf_csv, low_memory=False).fillna("")\n    needed = {"name", "latitude", "longitude", "type"}\n    if not needed.issubset(df.columns):\n        raise ValueError(f"Reference CSV missing columns: {sorted(needed - set(df.columns))}")\n    df["latitude"] = pd.to_numeric(df.latitude, errors="coerce")\n    df["longitude"] = pd.to_numeric(df.longitude, errors="coerce")\n    df = df.loc[df.latitude.between(41, 84) & df.longitude.between(-142, -52)].copy()\n    df["distance_km"] = [\n        _great_circle_km(center_lat, center_lon, float(r.latitude), float(r.longitude))\n        for _, r in df.iterrows()\n    ]\n    df = df.loc[df.distance_km <= radius_km].sort_values("distance_km").copy()\n    df["candidate_class"] = np.where(\n        df["type"].astype(str).str.fullmatch("hospital|hospitals", case=False, na=False),\n        "known_hospital_site", "other_healthcare_site_UNVERIFIED",\n    )\n    selected: list[pd.Series] = []\n    for category, cap in [("known_hospital_site", hospitals), ("other_healthcare_site_UNVERIFIED", controls)]:\n        count = 0\n        for _, row in df.loc[df.candidate_class.eq(category)].iterrows():\n            if count >= cap:\n                break\n            if all(_great_circle_km(float(row.latitude), float(row.longitude),\n                                    float(other.latitude), float(other.longitude)) * 1000 >= min_site_spacing_m\n                   for other in selected):\n                selected.append(row)\n                count += 1\n    if not selected:\n        raise RuntimeError("No candidate healthcare sites found near the selected city")\n    result = pd.DataFrame(selected).reset_index(drop=True)\n    result["site_id"] = result.apply(_site_token, axis=1)\n    return result\n\n\ndef prepare_review_chips_from_geotiff(\n    odhf_csv: str | Path, source_geotiff: str | Path, project_dir: str | Path,\n    imagery_license_url: str, imagery_source_name: str,\n    license_confirmed: bool = False,\n    center_lat: float = 45.4215, center_lon: float = -75.6972,\n    radius_km: float = 30, hospitals: int = 12, controls: int = 12,\n    image_size: int = 640, ground_width_m: float = 320,\n) -> pd.DataFrame:\n    """Offline imagery option: extract georeferenced chips from a licensed local GeoTIFF.\n\n    Raster must have genuine georeferencing and <=1m native GSD at pilot area.\n    This never generates automatic labels or training approval.\n    """\n    from rasterio.warp import reproject, Resampling\n    source_geotiff, project_dir = Path(source_geotiff), Path(project_dir)\n    if not license_confirmed or not imagery_license_url.startswith(\'https://\') or not imagery_source_name.strip():\n        raise ValueError(\'Source attribution, HTTPS license URL and explicit rights confirmation are required\')\n    if (project_dir / \'review_manifest.csv\').exists():\n        raise FileExistsError(\'Review manifest exists. Do not overwrite reviewed imagery or annotations\')\n    sites = select_review_sites(odhf_csv, center_lat, center_lon, radius_km, hospitals, controls)\n    rows, errors = [], []\n    with rasterio.open(source_geotiff) as source:\n        if source.crs is None or source.count < 3 or source.dtypes[0] != \'uint8\':\n            raise ValueError(\'Source requires >=3 uint8 RGB bands and a real CRS\')\n        for _, site in sites.iterrows():\n            lat, lon = float(site.latitude), float(site.longitude)\n            # Do not silently resample 10m satellite imagery and claim roof detail.\n            if source.crs.is_geographic:\n                gsd_m = max(abs(source.res[0]) * 111320 * math.cos(math.radians(lat)),\n                            abs(source.res[1]) * 111132)\n            else:\n                # Source CRS is projected, in meters for standard Ontario imagery.\n                unit = source.crs.linear_units_factor[1] if source.crs.linear_units_factor else 1\n                gsd_m = max(abs(source.res[0]), abs(source.res[1])) * unit\n            if gsd_m > 1.0:\n                raise ValueError(f\'Imagery native pixel size {gsd_m:.2f}m is too coarse for hospital rooftops\')\n            tile=f\'local_{site.site_id}_00\'\n            image_path=project_dir/\'review\'/\'images\'/f\'{tile}.png\'\n            tiff_path=project_dir/\'review\'/\'geotiffs\'/f\'{tile}.tif\'\n            ann_path=project_dir/\'review\'/\'annotations\'/f\'{tile}.json\'\n            bound=_mercator_extent(lat,lon,ground_width_m)\n            transform=from_bounds(*bound,width=image_size,height=image_size)\n            dest=np.zeros((3,image_size,image_size),dtype=np.uint8)\n            try:\n                for band in range(1,4):\n                    reproject(source=rasterio.band(source,band),destination=dest[band-1],\n                              src_transform=source.transform,src_crs=source.crs,\n                              dst_transform=transform,dst_crs=\'EPSG:3857\',\n                              resampling=Resampling.bilinear)\n                if np.mean(np.any(dest>0,axis=0)) < .97 or dest.std()<3:\n                    errors.append(f\'{tile}: outside source coverage / blank pixels\')\n                    continue\n                image_path.parent.mkdir(parents=True,exist_ok=True)\n                tiff_path.parent.mkdir(parents=True,exist_ok=True)\n                Image.fromarray(dest.transpose(1,2,0)).save(image_path)\n                with rasterio.open(tiff_path,\'w\',driver=\'GTiff\',width=image_size,height=image_size,\n                                   count=3,dtype=\'uint8\',crs=\'EPSG:3857\',transform=transform,\n                                   compress=\'deflate\') as dst:dst.write(dest)\n                rows.append({\n                    \'tile_id\':tile,\'site_id\':site.site_id,\'name\':str(site[\'name\']),\n                    \'candidate_class\':site.candidate_class,\'latitude\':lat,\'longitude\':lon,\n                    \'image\':str(image_path.relative_to(project_dir)),\n                    \'geotiff\':str(tiff_path.relative_to(project_dir)),\n                    \'annotation\':str(ann_path.relative_to(project_dir)),\n                    \'imagery_source\':imagery_source_name,\n                    \'imagery_license_url\':imagery_license_url,\n                    \'source_extent_3857\':json.dumps(bound),\'image_width\':image_size,\n                    \'image_height\':image_size,\'ground_width_m\':ground_width_m,\n                    \'review_status\':\'UNREVIEWED\',\'reviewed_label\':\'UNKNOWN\',\n                    \'reviewer\':\'\',\'source_checked\':\'NO\',\n                })\n            except (OSError,ValueError) as e:\n                errors.append(f\'{tile}: {str(e)[:240]}\')\n    project_dir.mkdir(parents=True,exist_ok=True)\n    (project_dir/\'imagery_errors.json\').write_text(json.dumps(errors,indent=2),encoding=\'utf-8\')\n    if not rows:\n        raise RuntimeError(\'No local GeoTIFF candidate chips overlap source. See imagery_errors.json\')\n    result=pd.DataFrame(rows,columns=COLUMN_NAMES)\n    result.to_csv(project_dir/\'review_manifest.csv\',index=False)\n    return result\n\n\ndef _mercator_extent(lat: float, lon: float, ground_width_m: float) -> tuple[float, float, float, float]:\n    if not (-85 < lat < 85 and -180 <= lon <= 180):\n        raise ValueError("Invalid latitude/longitude")\n    if not 60 <= ground_width_m <= 1000:\n        raise ValueError("Requested ground width must be 60..1000 meters")\n    center_x, center_y = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True).transform(lon, lat)\n    half = (ground_width_m / max(math.cos(math.radians(lat)), .01)) / 2\n    return center_x - half, center_y - half, center_x + half, center_y + half\n\n\ndef download_ontario_chip(\n    lat: float, lon: float, image_path: str | Path, geotiff_path: str | Path,\n    ground_width_m: float = 320, image_size: int = 640,\n    session: Any | None = None, service_url: str = ONTARIO_IMAGERY_SERVICE,\n) -> dict[str, Any]:\n    """Request one image from official Ontario ArcGIS ImageServer and georeference.\n\n    Uses REST exportImage f=json extent metadata. Fails rather than assuming an\n    image\'s geolocation or silently accepting transparent/no-data pixels.\n    """\n    if not (256 <= image_size <= 1024):\n        raise ValueError("Image size should be 256..1024 pixels")\n    if not service_url.startswith("https://") or "/ImageServer" not in service_url:\n        raise ValueError("Must use an explicit trusted HTTPS ArcGIS ImageServer URL")\n    http = session if session is not None else requests.Session()\n    bbox = _mercator_extent(float(lat), float(lon), float(ground_width_m))\n    result = http.get(service_url.rstrip("/") + "/exportImage", params={\n        "f": "json", "bbox": ",".join(f"{v:.3f}" for v in bbox),\n        "bboxSR": 3857, "imageSR": 3857, "size": f"{image_size},{image_size}",\n        "format": "png32", "adjustAspectRatio": "false",\n        "interpolation": "RSP_BilinearInterpolation",\n    }, timeout=80, headers={"User-Agent": "OSS4008-HospitalDetector-Educational/1.0"})\n    result.raise_for_status()\n    info = result.json()\n    if "error" in info:\n        raise RuntimeError(f"ArcGIS imagery service error: {info[\'error\']}")\n    extent = info.get("extent") or {}\n    sr = extent.get("spatialReference") or {}\n    if sr.get("wkid", sr.get("latestWkid")) not in (3857, 102100, 102113):\n        raise ValueError("Image service did not return the requested EPSG:3857 extent")\n    bounds = [extent.get(k) for k in ("xmin", "ymin", "xmax", "ymax")]\n    if any(not isinstance(v, (int, float)) or not math.isfinite(v) for v in bounds):\n        raise ValueError("ArcGIS response has no trustworthy geographic image extent")\n    url = info.get("href", "")\n    parsed = urlparse(url)\n    if parsed.scheme != "https" or not parsed.hostname:\n        raise ValueError("Image service did not return a safe HTTPS image link")\n    # An ArcGIS image export may use a temporary subdomain on the same provider.\n    if not (parsed.hostname.endswith("gov.on.ca") or parsed.hostname.endswith("arcgis.com")):\n        raise ValueError("Image export link host is outside expected Ontario/ArcGIS domains")\n    r = http.get(url, timeout=80, headers={"User-Agent": "OSS4008-HospitalDetector-Educational/1.0"})\n    r.raise_for_status()\n    if len(r.content) > 14_000_000:\n        raise ValueError("Unexpectedly large image export")\n    with Image.open(BytesIO(r.content)) as raw:\n        if raw.size != (image_size, image_size):\n            raise ValueError(f"ImageServer returned {raw.size}, expected {(image_size, image_size)}")\n        if "A" in raw.getbands():\n            alpha = np.asarray(raw.getchannel("A"))\n            if np.mean(alpha > 0) < .97:\n                raise ValueError("Imagery is mostly transparent: likely outside service coverage")\n        img = raw.convert("RGB")\n    rgb = np.asarray(img)\n    if rgb.std() < 3:\n        raise ValueError("Image export appears blank or nearly uniform")\n    image_path, geotiff_path = Path(image_path), Path(geotiff_path)\n    image_path.parent.mkdir(parents=True, exist_ok=True)\n    geotiff_path.parent.mkdir(parents=True, exist_ok=True)\n    img.save(image_path)\n    transform = from_bounds(*bounds, width=image_size, height=image_size)\n    with rasterio.open(geotiff_path, "w", driver="GTiff", width=image_size, height=image_size,\n                       count=3, dtype="uint8", crs="EPSG:3857", transform=transform,\n                       compress="deflate") as dest:\n        dest.write(rgb.transpose(2, 0, 1))\n    return {"image": str(image_path), "geotiff": str(geotiff_path),\n            "extent": bounds, "size": image_size, "ground_width_m": ground_width_m,\n            "service": service_url, "license_url": IMAGERY_LICENSE_URL}\n\n\ndef prepare_review_chips(\n    odhf_csv: str | Path, project_dir: str | Path, center_lat: float = 45.4215,\n    center_lon: float = -75.6972, radius_km: float = 30,\n    hospitals: int = 12, controls: int = 12, image_size: int = 640,\n    ground_width_m: float = 320, session: Any | None = None,\n) -> pd.DataFrame:\n    """Acquire chips & manifest. Does NOT create approved annotations."""\n    project_dir = Path(project_dir)\n    project_dir.mkdir(parents=True, exist_ok=True)\n    sites = select_review_sites(odhf_csv, center_lat, center_lon, radius_km, hospitals, controls)\n    # RESUME SAFE: never overwrite existing human review or annotation files.\n    manifest_path = project_dir / "review_manifest.csv"\n    rows: list[dict[str, Any]] = []\n    if manifest_path.is_file():\n        existing = pd.read_csv(manifest_path, keep_default_na=False)\n        if list(existing.columns) != COLUMN_NAMES:\n            raise ValueError("Existing review manifest schema differs; refusing overwrite")\n        rows = existing.to_dict(\'records\')\n    existing_by_site = {str(r[\'site_id\']): j for j, r in enumerate(rows)}\n    errors: list[str] = []\n    for i, s in sites.iterrows():\n        prior_index = existing_by_site.get(str(s.site_id))\n        if prior_index is not None:\n            prior = rows[prior_index]\n            # Approved/rejected human decisions are immutable during download repair.\n            if str(prior.get(\'review_status\')) in (\'APPROVED\',\'REJECTED\'):\n                continue\n            existing_image = project_dir / str(prior.get(\'image\',\'\'))\n            existing_tiff = project_dir / str(prior.get(\'geotiff\',\'\'))\n            if existing_image.is_file() and existing_tiff.is_file():\n                continue\n            # Only an unreviewed missing-file entry can be repaired in place.\n        tile = f"ontario_{s.site_id}_00"\n        image = project_dir / "review" / "images" / f"{tile}.png"\n        tiff = project_dir / "review" / "geotiffs" / f"{tile}.tif"\n        annotation = project_dir / "review" / "annotations" / f"{tile}.json"\n        try:\n            info = download_ontario_chip(float(s.latitude), float(s.longitude), image, tiff,\n                                         ground_width_m, image_size, session=session)\n        except (requests.RequestException, RuntimeError, ValueError, OSError) as error:\n            errors.append(f"{tile}: {str(error)[:220]}")\n            continue\n        # Deliberately no blank LabelMe JSON auto-created. An empty annotation\n        # could otherwise be confused with a human-verified negative example.\n        candidate={\n            "tile_id": tile, "site_id": s.site_id, "name": str(s["name"]),\n            "candidate_class": s.candidate_class, "latitude": s.latitude, "longitude": s.longitude,\n            "image": str(image.relative_to(project_dir)), "geotiff": str(tiff.relative_to(project_dir)),\n            "annotation": str(annotation.relative_to(project_dir)),\n            "imagery_source": info["service"], "imagery_license_url": info["license_url"],\n            "source_extent_3857": json.dumps(info["extent"]),\n            "image_width": image_size, "image_height": image_size, "ground_width_m": ground_width_m,\n            "review_status": "UNREVIEWED", "reviewed_label": "UNKNOWN", "reviewer": "", "source_checked": "NO",\n        }\n        if prior_index is None:\n            rows.append(candidate)\n            existing_by_site[str(s.site_id)] = len(rows)-1\n        else:\n            rows[prior_index] = candidate  # previously UNREVIEWED, no human annotation lost\n        # Save each completed chip. An interrupted 100-site acquisition must not\n        # discard the 99 previously successful downloads or their metadata.\n        checkpoint = pd.DataFrame(rows, columns=COLUMN_NAMES)\n        checkpoint_tmp = manifest_path.with_suffix(\'.csv.checkpoint\')\n        checkpoint.to_csv(checkpoint_tmp, index=False)\n        checkpoint_tmp.replace(manifest_path)\n        if len(rows)%10 == 0:\n            print(f"Saved {len(rows)} aerial chips in Drive (includes earlier images); current site {tile}")\n    (project_dir / "imagery_errors.json").write_text(json.dumps(errors, indent=2), encoding="utf-8")\n    if not rows:\n        raise RuntimeError("No imagery downloaded; inspect imagery_errors.json for provider/coverage details")\n    manifest = pd.DataFrame(rows, columns=COLUMN_NAMES)\n    temp = manifest_path.with_suffix(\'.csv.pending\')\n    manifest.to_csv(temp, index=False)\n    temp.replace(manifest_path)\n    return manifest\n\n\ndef labelme_rectangle_to_yolo(points: Any, width: int, height: int) -> tuple[float, float, float, float]:\n    if len(points) != 2 or width <= 0 or height <= 0:\n        raise ValueError("LabelMe hospital rectangle must have two corners and a valid image size")\n    (x1, y1), (x2, y2) = points\n    x1, x2 = sorted((max(0, min(width, float(x1))), max(0, min(width, float(x2)))))\n    y1, y2 = sorted((max(0, min(height, float(y1))), max(0, min(height, float(y2)))))\n    if x2 <= x1 or y2 <= y1:\n        raise ValueError("Degenerate bounding box")\n    return ((x1 + x2) / (2 * width), (y1 + y2) / (2 * height),\n            (x2 - x1) / width, (y2 - y1) / height)\n\n\ndef _verified_annotation(annotation: Path, image: Path, verified_label: str) -> list[str]:\n    if not annotation.is_file():\n        raise ValueError(f"Missing independently reviewed LabelMe annotation: {annotation}")\n    doc = json.loads(annotation.read_text(encoding="utf-8"))\n    with Image.open(image) as img:\n        width, height = img.size\n    if int(doc.get("imageWidth", -1)) != width or int(doc.get("imageHeight", -1)) != height:\n        raise ValueError(f"Annotation image dimensions mismatch for {image.name}")\n    if doc.get("flags", {}).get("verified") is not True:\n        raise ValueError(f"LabelMe file is missing explicit verified=true flag: {annotation}")\n    boxes = []\n    for shape in doc.get("shapes", []):\n        if str(shape.get("label", "")).casefold() != "hospital":\n            raise ValueError(f"Unsupported or ambiguous LabelMe shape label: {shape.get(\'label\')}")\n        if shape.get("shape_type") != "rectangle":\n            raise ValueError(f"Expected rectangle annotation: {annotation.name}")\n        yolo = labelme_rectangle_to_yolo(shape.get("points", []), width, height)\n        boxes.append("0 " + " ".join(f"{v:.7f}" for v in yolo))\n    if verified_label == "HOSPITAL_PRESENT" and not boxes:\n        raise ValueError("Positive, approved image must have at least one verified hospital box")\n    if verified_label == "NO_HOSPITAL" and boxes:\n        raise ValueError("Reviewed negative image cannot contain a hospital box")\n    if verified_label not in ("HOSPITAL_PRESENT", "NO_HOSPITAL"):\n        raise ValueError("reviewed_label must explicitly identify verified positive or negative")\n    return boxes\n\n\ndef split_verified_dataset(\n    project_dir: str | Path, seed: int = 4008, min_sites_per_class: int = 3,\n) -> dict[str, Any]:\n    """Build a YOLO dataset with no site leakage and independently reviewed labels.\n\n    Requires at least three separately reviewed positive *sites* and three\n    negative *sites*, one per class in each train/val/test split. Three sites\n    per class is a mechanical minimum; meaningful accuracy requires many more.\n    """\n    project_dir = Path(project_dir)\n    manifest = project_dir / "review_manifest.csv"\n    if not manifest.is_file():\n        raise FileNotFoundError(f"Missing candidate review manifest: {manifest}")\n    df = pd.read_csv(manifest, keep_default_na=False)\n    missing = [k for k in COLUMN_NAMES if k not in df.columns]\n    if missing:\n        raise ValueError(f"Manifest missing columns: {missing}")\n    # Pending/auto-draft-only images are NOT training labels. Excluding them\n    # avoids forcing the reviewer to clear 100 candidates before training on\n    # a small, independently approved pilot subset. Never silently promote.\n    if len(df) == 0 or not df.review_status.isin(["APPROVED", "REJECTED", "UNREVIEWED"]).all():\n        raise ValueError("Invalid review status; only APPROVED images are eligible for training")\n    if df.tile_id.duplicated().any():\n        raise ValueError("Duplicate tile_id present")\n    df = df.loc[df.review_status.eq("APPROVED")].copy()\n    if df.empty or not df.source_checked.eq("YES").all() or df.reviewer.astype(str).str.strip().eq("").any():\n        raise ValueError("At least one APPROVED image and named human reviewer/source check required")\n    prepared = []\n    for _, row in df.iterrows():\n        img = (project_dir / row.image).resolve()\n        ann = (project_dir / row.annotation).resolve()\n        if not img.is_relative_to(project_dir.resolve()) or not ann.is_relative_to(project_dir.resolve()):\n            raise ValueError("Manifest contains an unsafe path outside the project folder")\n        if not img.is_file():\n            raise FileNotFoundError(f"Missing reviewed image: {img}")\n        boxes = _verified_annotation(ann, img, str(row.reviewed_label))\n        prepared.append({"tile_id": row.tile_id, "site_id": row.site_id,\n                         "category": row.reviewed_label, "image": img,\n                         "boxes": boxes, "reviewer": row.reviewer})\n    # Group by independent hospital/site; not by neighboring image crops.\n    grouped: dict[str, list[dict]] = {}\n    for item in prepared:\n        grouped.setdefault(item["site_id"], []).append(item)\n    classes = {"HOSPITAL_PRESENT": [], "NO_HOSPITAL": []}\n    for sid, items in grouped.items():\n        cats = {x["category"] for x in items}\n        # When one campus has mixed positive/negative patches, keep the entire\n        # campus together and count it as a positive group.\n        primary = "HOSPITAL_PRESENT" if "HOSPITAL_PRESENT" in cats else "NO_HOSPITAL"\n        classes[primary].append(sid)\n    if any(len(v) < max(3, min_sites_per_class) for v in classes.values()):\n        raise ValueError("Insufficient independent APPROVED positive/negative sites for site-disjoint train/val/test. "\n                         f"Found: { {k: len(v) for k, v in classes.items()} }")\n    rng = random.Random(seed)\n    assignment: dict[str, str] = {}\n    for group_names in classes.values():\n        group_names = sorted(group_names)\n        rng.shuffle(group_names)\n        # Always reserve an independent positive/negative group for val and test.\n        test_id, val_id = group_names[0], group_names[1]\n        assignment[test_id] = "test"\n        assignment[val_id] = "val"\n        for sid in group_names[2:]:\n            assignment[sid] = "train"\n    output = project_dir / "hospital_yolo"\n    if output.exists():\n        # Prevent silent reuse of stale images/labels when dataset changes.\n        shutil.rmtree(output)\n    counts = {"train": {"positive": 0, "negative": 0},\n              "val": {"positive": 0, "negative": 0},\n              "test": {"positive": 0, "negative": 0}}\n    assignment_rows = []\n    for item in prepared:\n        split = assignment[item["site_id"]]\n        dest_image = output / "images" / split / item["image"].name\n        dest_label = output / "labels" / split / (item["image"].stem + ".txt")\n        dest_image.parent.mkdir(parents=True, exist_ok=True)\n        dest_label.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(item["image"], dest_image)\n        dest_label.write_text("\\n".join(item["boxes"]) + ("\\n" if item["boxes"] else ""), encoding="utf-8")\n        counts[split]["positive" if item["boxes"] else "negative"] += 1\n        assignment_rows.append({"tile_id": item["tile_id"], "site_id": item["site_id"],\n                                "split": split, "reviewed_label": item["category"],\n                                "image_sha256": sha256(item["image"])})\n    if any(value <= 0 for v in counts.values() for value in v.values()):\n        raise ValueError("Need verified positive and negative image examples in each train/val/test split")\n    config = {"path": str(output.resolve()), "train": "images/train", "val": "images/val",\n              "test": "images/test", "names": {0: "hospital"}}\n    yaml_path = output / "data.yaml"\n    yaml_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")\n    pd.DataFrame(assignment_rows).to_csv(output / "site_split_manifest.csv", index=False)\n    sources = sorted({str(v) for v in df.imagery_source if str(v).strip()})\n    licenses = sorted({str(v) for v in df.imagery_license_url if str(v).strip()})\n    if len(sources) == 0 or len(licenses) == 0:\n        raise ValueError("Reviewed imagery must include recorded sources and licence URLs")\n    provenance = {"status": "ANNOTATIONS_REVIEWED", "label_type": "LabelMe rectangles",\n                  "image_sources": sources, "license_urls": licenses,\n                  "source_manifest_sha256": sha256(manifest),\n                  "split_manifest_sha256": sha256(output / "site_split_manifest.csv"),\n                  "site_disjoint": True, "seed": seed, "counts": counts,\n                  "notes": "Site disjoint does not guarantee cross-city generalization; inspect data distribution."}\n    (output / "dataset_provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")\n    return {"dataset_yaml": str(yaml_path), "counts": counts,\n            "num_sites": len(grouped), "site_disjoint": True}\n\n\ndef train_and_evaluate(\n    project_dir: str | Path, epochs: int = 40, image_size: int = 640,\n    batch: int = 8, pretrained_model: str = "yolo11n.pt",\n) -> dict[str, Any]:\n    """Actually fine-tune YOLO on hospital labels, evaluate held-out sites.\n\n    This is a real computational training step, not executed unless approved\n    imagery/labels were provided, split_verified_dataset succeeded, and the\n    Ultralytics package/model weights can be downloaded in a Colab runtime.\n    """\n    project_dir = Path(project_dir)\n    data = project_dir / "hospital_yolo" / "data.yaml"\n    provenance = project_dir / "hospital_yolo" / "dataset_provenance.json"\n    if not data.is_file() or not provenance.is_file():\n        raise RuntimeError("BLOCKED: Run split_verified_dataset() on real reviewed annotations first")\n    info = json.loads(provenance.read_text(encoding="utf-8"))\n    if info.get("status") != "ANNOTATIONS_REVIEWED" or info.get("site_disjoint") is not True:\n        raise RuntimeError("Dataset has no verified site-disjoint training provenance")\n    current_split = project_dir / "hospital_yolo" / "site_split_manifest.csv"\n    if (not current_split.is_file() or\n        info.get("split_manifest_sha256") != sha256(current_split) or\n        info.get("source_manifest_sha256") != sha256(project_dir / "review_manifest.csv")):\n        raise RuntimeError("The reviewed dataset changed after conversion; rebuild the site-disjoint dataset")\n    if not 1 <= epochs <= 250 or not 256 <= image_size <= 1280 or not 1 <= batch <= 64:\n        raise ValueError("Invalid training hyperparameters")\n    from ultralytics import YOLO  # defer heavy import until real training\n    import torch\n    model = YOLO(pretrained_model)\n    device = 0 if torch.cuda.is_available() else "cpu"\n    fit = model.train(data=str(data), epochs=epochs, imgsz=image_size, batch=batch,\n                      seed=4008, deterministic=True, patience=12,\n                      device=device, workers=2, project=str(project_dir / "runs"),\n                      name="hospital_detector", exist_ok=True)\n    weights = Path(fit.save_dir) / "weights" / "best.pt"\n    if not weights.is_file():\n        raise RuntimeError("Training ended without a best.pt checkpoint")\n    detector_dir = project_dir / "detector"\n    detector_dir.mkdir(parents=True, exist_ok=True)\n    model_path = detector_dir / "hospital_best.pt"\n    shutil.copy2(weights, model_path)\n    evaluator = YOLO(str(model_path))\n    metrics = evaluator.val(data=str(data), split="test", imgsz=image_size, device=device,\n                            project=str(project_dir / "runs"), name="heldout_test", exist_ok=True)\n    values = {"precision": float(metrics.box.mp), "recall": float(metrics.box.mr),\n              "mAP50": float(metrics.box.map50), "mAP50_95": float(metrics.box.map)}\n    report = {"status": "TRAINED_AND_EVALUATED", "model": str(model_path),\n              "weights_sha256": sha256(model_path), "dataset_yaml": str(data),\n              "image_size": image_size, "batch": batch, "epochs_requested": epochs,\n              "device": str(device), "pretrained_model": pretrained_model,\n              "test_metrics": values, "data_provenance": str(provenance),\n              "source_manifest_sha256": info["source_manifest_sha256"],\n              "split_manifest_sha256": info["split_manifest_sha256"],\n              "limit": "Hospital rooftop appearance alone cannot establish building function; generalization requires independent sites/cities."}\n    (detector_dir / "training_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    return report\n\n\ndef _nms(boxes: list[dict[str, Any]], iou_limit: float = 0.5) -> list[dict[str, Any]]:\n    """Greedy cross-tile NMS for axis-aligned boxes, highest model score first."""\n    def iou(a, b):\n        x1, y1 = max(a["box"][0], b["box"][0]), max(a["box"][1], b["box"][1])\n        x2, y2 = min(a["box"][2], b["box"][2]), min(a["box"][3], b["box"][3])\n        intersection = max(0, x2-x1) * max(0, y2-y1)\n        aa = (a["box"][2]-a["box"][0]) * (a["box"][3]-a["box"][1])\n        ab = (b["box"][2]-b["box"][0]) * (b["box"][3]-b["box"][1])\n        return intersection / (aa + ab - intersection) if aa + ab > intersection else 0.0\n    chosen = []\n    for candidate in sorted(boxes, key=lambda x: -x["confidence"]):\n        if all(iou(candidate, other) < iou_limit for other in chosen):\n            chosen.append(candidate)\n    return chosen\n\n\ndef _tile_starts(length: int, size: int, overlap: float):\n    if length <= size:\n        return [0]\n    stride = max(1, int(size * (1 - overlap)))\n    return sorted(set(list(range(0, length - size + 1, stride)) + [length - size]))\n\n\ndef export_geotiff_predictions(\n    project_dir: str | Path, geotiff: str | Path, output_geojson: str | Path,\n    confidence: float = .35, tile_size: int = 640, overlap: float = .2,\n    predictor: Any | None = None,\n) -> dict[str, Any]:\n    """Export *true trained* hospital model boxes as WGS84 points for existing UI.\n\n    `predictor` is injectable solely for offline coordinate-contract tests; real\n    inference requires matching trained checkpoint and training_report.json.\n    """\n    project_dir, geotiff, output_geojson = Path(project_dir), Path(geotiff), Path(output_geojson)\n    report_file = project_dir / "detector" / "training_report.json"\n    if predictor is None:\n        if not report_file.is_file():\n            raise RuntimeError("BLOCKED: no evaluated hospital model report. Train and evaluate first.")\n        report = json.loads(report_file.read_text(encoding="utf-8"))\n        model_file = project_dir / "detector" / "hospital_best.pt"\n        if (report.get("status") != "TRAINED_AND_EVALUATED" or not model_file.is_file() or\n            report.get("weights_sha256") != sha256(model_file) or\n            not (project_dir / "hospital_yolo" / "site_split_manifest.csv").is_file() or\n            report.get("split_manifest_sha256") != sha256(project_dir / "hospital_yolo" / "site_split_manifest.csv") or\n            report.get("source_manifest_sha256") != sha256(project_dir / "review_manifest.csv")):\n            raise RuntimeError("Missing, stale or mismatched trained hospital detector checkpoint / dataset")\n        from ultralytics import YOLO\n        predictor = YOLO(str(model_file))\n        if "hospital" not in [str(n).casefold() for n in predictor.names.values()]:\n            raise RuntimeError("Model weights are not for the hospital class")\n        model_id = report["weights_sha256"]\n    else:\n        # Injected predictors are not AI evidence and their exports cannot be\n        # passed off as real predictions.\n        model_id = "TEST_PREDICTOR_DO_NOT_PUBLISH"\n    if not 0 < confidence < 1 or not 256 <= tile_size <= 1280 or not 0 <= overlap < .5:\n        raise ValueError("Invalid prediction tiling/confidence configuration")\n    with rasterio.open(geotiff) as src:\n        if src.crs is None or src.count < 3 or src.dtypes[0] != "uint8":\n            raise ValueError("Need georeferenced 3-band uint8 GeoTIFF for inference")\n        if src.width > 30000 or src.height > 30000:\n            raise ValueError("Input too large for in-memory inference; split into separate geotiffs")\n        rgb = src.read([1, 2, 3]).transpose(1, 2, 0)\n        transform = src.transform\n        crs = src.crs\n    boxes = []\n    for y in _tile_starts(rgb.shape[0], tile_size, overlap):\n        for x in _tile_starts(rgb.shape[1], tile_size, overlap):\n            tile = rgb[y:y+tile_size, x:x+tile_size]\n            results = predictor.predict(source=Image.fromarray(tile), conf=confidence, verbose=False)\n            for result in results:\n                for b in result.boxes:\n                    if int(b.cls[0]) != 0:\n                        continue\n                    x1, y1, x2, y2 = (float(v) for v in b.xyxy[0].tolist())\n                    x1, x2 = max(0, x+x1), min(rgb.shape[1], x+x2)\n                    y1, y2 = max(0, y+y1), min(rgb.shape[0], y+y2)\n                    if x2 <= x1 or y2 <= y1:\n                        continue\n                    boxes.append({"box": (x1, y1, x2, y2), "confidence": float(b.conf[0])})\n    boxes = _nms(boxes)\n    features = []\n    for b in boxes:\n        x1, y1, x2, y2 = b["box"]\n        map_x, map_y = transform * ((x1 + x2) / 2, (y1 + y2) / 2)\n        lon, lat = transform_coordinates(crs, "EPSG:4326", [map_x], [map_y])\n        features.append({"type": "Feature", "geometry": {"type": "Point", "coordinates": [lon[0], lat[0]]},\n                         "properties": {"class": "hospital", "confidence": b["confidence"],\n                                        "bbox_pixels": [x1, y1, x2, y2],\n                                        "method": "DETECTOR_TEST_ONLY" if model_id.startswith("TEST_") else "trained_YOLO_hospital_building_detector",\n                                        "weights_sha256": model_id, "source_geotiff": geotiff.name}})\n    collection = {"type": "FeatureCollection", "features": features}\n    if model_id.startswith("TEST_"):\n        # Only return test data to caller, never save a fake model export.\n        return collection\n    output_geojson.parent.mkdir(parents=True, exist_ok=True)\n    output_geojson.write_text(json.dumps(collection, indent=2), encoding="utf-8")\n    return collection\n\n\ndef make_candidate_contact_sheet(project_dir: str | Path, width: int = 900) -> Path:\n    """Create a review sheet for human labeling; not an AI result."""\n    root = Path(project_dir)\n    df = pd.read_csv(root / "review_manifest.csv", keep_default_na=False)\n    thumbs = []\n    for _, r in df.iterrows():\n        img_path = root / r.image\n        with Image.open(img_path) as src:\n            im = ImageOps.fit(src.convert("RGB"), (width // 3, width // 3))\n            thumbs.append((im.copy(), f"{r.tile_id[:22]}\\n{r.candidate_class}"))\n    w = width\n    h = (width // 3 + 54) * math.ceil(len(thumbs) / 3)\n    sheet = Image.new("RGB", (w, max(h, 1)), "white")\n    draw = ImageDraw.Draw(sheet)\n    for i, (image, label) in enumerate(thumbs):\n        x = (i % 3) * (width // 3)\n        y = (i // 3) * (width // 3 + 54)\n        sheet.paste(image, (x, y))\n        draw.multiline_text((x + 5, y + width // 3 + 5), label, fill="black", spacing=3)\n    target = root / "review_contact_sheet.jpg"\n    sheet.save(target, quality=90)\n    return target\n\n\ndef review_workspace_status(project_dir: str | Path) -> dict[str, Any]:\n    """Read-only live inspection; never report imagery as downloaded without files."""\n    project_dir = Path(project_dir)\n    result: dict[str, Any] = {"manifest_found": False, "records": 0,\n        "images_found": 0, "geotiffs_found": 0, "annotations_found": 0,\n        "review_status": {}, "missing_files": [], "download_errors": []}\n    errors = project_dir / "imagery_errors.json"\n    if errors.is_file():\n        try:\n            result["download_errors"] = json.loads(errors.read_text(encoding="utf-8"))\n        except (json.JSONDecodeError, OSError):\n            result["download_errors"] = ["Imagery error log could not be parsed"]\n    manifest = project_dir / "review_manifest.csv"\n    if not manifest.is_file():\n        return result\n    df = pd.read_csv(manifest, keep_default_na=False)\n    result["manifest_found"] = True\n    result["records"] = len(df)\n    if "review_status" in df:\n        result["review_status"] = {str(k): int(v) for k, v in df.review_status.value_counts().items()}\n    for field, key in (("image", "images_found"), ("geotiff", "geotiffs_found"),\n                       ("annotation", "annotations_found")):\n        if field not in df:\n            result["missing_files"].append(f"Missing column {field}")\n            continue\n        for loc, value in enumerate(df[field]):\n            path = (project_dir / str(value)).resolve()\n            if path.is_relative_to(project_dir.resolve()) and path.is_file():\n                result[key] += 1\n            else:\n                # Unreviewed/rejected images SHOULD NOT have annotation JSON.\n                # Only missing imagery or missing approved annotations are blockers.\n                row_status = str(df.iloc[loc]["review_status"]) if "review_status" in df.columns else ""\n                if field == "annotation" and row_status in ("UNREVIEWED", "REJECTED"):\n                    continue\n                if len(result["missing_files"]) < 12:\n                    result["missing_files"].append(str(value))\n    return result\n\n\ndef evaluated_model_ready(project_dir: str | Path) -> tuple[bool, str]:\n    """Check an actual checkpoint against an evaluated, matching reviewed dataset."""\n    root = Path(project_dir)\n    report_path = root / "detector" / "training_report.json"\n    model_path = root / "detector" / "hospital_best.pt"\n    if not report_path.is_file() or not model_path.is_file():\n        return False, "No evaluated hospital detector checkpoint exists"\n    try:\n        report = json.loads(report_path.read_text(encoding="utf-8"))\n        splits = root / "hospital_yolo" / "site_split_manifest.csv"\n        original = root / "review_manifest.csv"\n        if (report.get("status") != "TRAINED_AND_EVALUATED" or\n            report.get("weights_sha256") != sha256(model_path) or\n            not splits.is_file() or report.get("split_manifest_sha256") != sha256(splits) or\n            not original.is_file() or report.get("source_manifest_sha256") != sha256(original)):\n            return False, "Model report or reviewed dataset does not match checkpoint"\n    except (OSError, ValueError, TypeError, KeyError) as exc:\n        return False, f"Model provenance could not be verified: {exc}"\n    return True, "Evaluated model checkpoint and dataset hashes verified"\n\n\ndef export_all_heldout_predictions(project_dir: str | Path, output_geojson: str | Path,\n                                    confidence: float = .35, max_sites: int = 40) -> dict[str, Any]:\n    """Run real model inference on all independent test sites and atomically save map GeoJSON.\n\n    Never substitutes reference coordinates for detector predictions. Each feature\n    inherits the SHA-256 fingerprint of the evaluated model weights.\n    """\n    project_dir = Path(project_dir)\n    ready, reason = evaluated_model_ready(project_dir)\n    if not ready:\n        raise RuntimeError("Cannot export actual detections: " + reason)\n    split_file = project_dir / "hospital_yolo" / "site_split_manifest.csv"\n    splits = pd.read_csv(split_file, keep_default_na=False)\n    if not {"tile_id", "site_id", "split"}.issubset(splits):\n        raise ValueError("Invalid site split manifest")\n    selected = splits.loc[splits.split.eq("test")]\n    if selected.empty or selected.site_id.nunique() > max_sites:\n        raise ValueError("No valid held-out test sites, or too many sites")\n    review = pd.read_csv(project_dir / "review_manifest.csv", keep_default_na=False)\n    if not {"tile_id", "geotiff", "site_id"}.issubset(review):\n        raise ValueError("Review manifest lacks georeferenced imagery paths")\n    combined = selected.merge(review[["tile_id", "geotiff", "site_id"]], on=["tile_id", "site_id"], validate="one_to_one")\n    if len(combined) != len(selected):\n        raise ValueError("Missing test imagery: refused partial map export")\n    output_geojson = Path(output_geojson)\n    features = []\n    for _, row in combined.iterrows():\n        tif = (project_dir / row.geotiff).resolve()\n        if not tif.is_relative_to(project_dir.resolve()) or not tif.is_file():\n            raise FileNotFoundError(f"Held-out source GeoTIFF unavailable: {row.geotiff}")\n        single = export_geotiff_predictions(project_dir, tif, output_geojson.with_suffix(\'.temporary.geojson\'), confidence=confidence)\n        for f in single["features"]:\n            f["properties"]["test_site_id"] = str(row.site_id)\n            features.append(f)\n    if len(features) > 3000:\n        raise ValueError("Too many detections for the existing map uploader")\n    geojson = {"type": "FeatureCollection", "features": features}\n    output_geojson.parent.mkdir(parents=True, exist_ok=True)\n    tmp = output_geojson.with_suffix(\'.incomplete\')\n    tmp.write_text(json.dumps(geojson, indent=2), encoding="utf-8")\n    tmp.replace(output_geojson)\n    interim = output_geojson.with_suffix(\'.temporary.geojson\')\n    if interim.is_file():\n        interim.unlink()\n    return {"status": "EXPORTED_EVALUATED_MODEL_OUTPUT", "test_sites": int(selected.site_id.nunique()),\n            "features": len(features), "geojson": str(output_geojson)}\n\n\ndef infer_in_selected_circle(\n    project_dir: str | Path, latitude: float, longitude: float, radius_m: float,\n    confidence: float = .35, max_images: int = 16,\n    session: Any | None = None,\n) -> dict[str, Any]:\n    """Evaluate the actual trained detector inside a small drawn Ontario study circle.\n\n    Backend for the already existing Colab map assistant. No new map widget or\n    control is needed. A ≤250 m radius prevents attempting city-wide imagery\n    acquisition in a free Colab environment. Returns only real model outputs;\n    untrained reference records are never considered detections.\n    """\n    from tempfile import TemporaryDirectory\n    project_dir=Path(project_dir)\n    good, reason=evaluated_model_ready(project_dir)\n    if not good:\n        raise RuntimeError(\'Hospital inference unavailable: \'+reason)\n    latitude,longitude,radius_m=float(latitude),float(longitude),float(radius_m)\n    if not (math.isfinite(latitude) and math.isfinite(longitude) and math.isfinite(radius_m)):\n        raise ValueError(\'Invalid study circle coordinates\')\n    if not (41.7 <= latitude <= 57 and -95.2 <= longitude <= -73.8):\n        raise ValueError(\'The current rights-verified imagery workflow covers Ontario pilot areas only\')\n    if not (20 <= radius_m <= 250):\n        raise ValueError(\'Draw a study circle with radius 20–250 metres for AI inference\')\n    if not (0 < confidence < 1 and 1 <= max_images <= 16):\n        raise ValueError(\'Invalid inference configuration\')\n    converter = Transformer.from_crs(\'EPSG:4326\',\'EPSG:3857\',always_xy=True)\n    inverse=Transformer.from_crs(\'EPSG:3857\',\'EPSG:4326\',always_xy=True)\n    center_x,center_y=converter.transform(longitude,latitude)\n    step_m=250.0\n    xy_centres=[(0.0,0.0)] if radius_m <= 130 else [\n        (dx*step_m,dy*step_m)\n        for dx in (-1,0,1) for dy in (-1,0,1)\n        if math.hypot(dx*step_m,dy*step_m) <= radius_m + 160\n    ]\n    if len(xy_centres)>max_images:\n        raise RuntimeError(\'Study circle exceeds capped aerial imagery requests\')\n    features=[]\n    errors=[]\n    with TemporaryDirectory(prefix=\'oss4008_area_\') as temp:\n        temp_path=Path(temp)\n        for idx,(dx,dy) in enumerate(xy_centres):\n            # EPSG:3857 coordinates are stretched by 1/cos(latitude), so scale\n            # physically measured offsets when moving the tile centres.\n            lon,lat=inverse.transform(center_x+dx/max(math.cos(math.radians(latitude)),.01),\n                                      center_y+dy/max(math.cos(math.radians(latitude)),.01))\n            image=temp_path/f\'chip_{idx:02d}.png\'\n            tif=temp_path/f\'chip_{idx:02d}.tif\'\n            try:\n                download_ontario_chip(lat,lon,image,tif,session=session)\n                geo=export_geotiff_predictions(project_dir,tif,temp_path/f\'partial_{idx:02d}.geojson\',confidence=confidence)\n                for feature in geo[\'features\']:\n                    f_lat=feature[\'geometry\'][\'coordinates\'][1]\n                    f_lon=feature[\'geometry\'][\'coordinates\'][0]\n                    if _great_circle_km(latitude,longitude,f_lat,f_lon)*1000 <= radius_m:\n                        features.append(feature)\n            except Exception as exc:\n                errors.append(f\'Tile {idx}: {type(exc).__name__}: {str(exc)[:170]}\')\n    if errors:\n        raise RuntimeError(f\'Incomplete imagery/inference for study circle ({len(errors)} of {len(xy_centres)} tiles): \'+ \'; \'.join(errors[:2]))\n    # Nearby predicted centres often duplicate across overlapping orthophoto\n    # chips. Keep the strongest score within 12 metres; NOT ODHF data.\n    unique=[]\n    for feature in sorted(features,key=lambda f:-float(f[\'properties\'][\'confidence\'])):\n        lon,lat=feature[\'geometry\'][\'coordinates\']\n        if all(_great_circle_km(lat,lon,u[\'geometry\'][\'coordinates\'][1],u[\'geometry\'][\'coordinates\'][0])*1000>12\n               for u in unique):\n            unique.append(feature)\n    if len(unique)>3000:\n        raise RuntimeError(\'Excessive inference features; inspect detector quality\')\n    report=json.loads((project_dir/\'detector\'/\'training_report.json\').read_text(encoding=\'utf-8\'))\n    result={\'type\':\'FeatureCollection\',\'features\':unique}\n    output_dir=project_dir/\'predictions\'\n    output_dir.mkdir(parents=True,exist_ok=True)\n    out=output_dir/f\'circle_{latitude:.4f}_{longitude:.4f}_{round(radius_m)}m.geojson\'\n    out.write_text(json.dumps(result,indent=2),encoding=\'utf-8\')\n    return {\'ok\':True,\'features\':unique,\'count\':len(unique),\n            \'image_chips\':len(xy_centres),\'model_sha256\':report[\'weights_sha256\'],\n            \'saved_to\':str(out), \'note\':\'Screening candidates, not verified hospital identity\'}\n'
HOSPITAL_REVIEW_SOURCE = '"""Human annotation helper for the *separate* hospital AI training notebook.\n\nThis does NOT modify the working Hospital Infrastructure Canada map UI. It\nrequires a person to verify each selected roof and its building function.\n"""\nfrom __future__ import annotations\nimport base64\nimport html\nimport json\nimport uuid\nfrom pathlib import Path\nfrom typing import Any\n\nimport pandas as pd\nfrom PIL import Image\nfrom IPython.display import HTML, Javascript, JSON, display\n\n\ndef save_annotation_review(root: str | Path, payload: dict[str, Any]) -> dict[str, Any]:\n    root = Path(root).resolve()\n    csv_path = root / "review_manifest.csv"\n    manifest = pd.read_csv(csv_path, keep_default_na=False)\n    token = str(payload.get("tile_id", ""))\n    indexes = manifest.index[manifest.tile_id.eq(token)].tolist()\n    if len(indexes) != 1:\n        raise ValueError("Unknown tile ID")\n    index = indexes[0]\n    row = manifest.loc[index]\n    image_path = (root / row.image).resolve()\n    annotation_path = (root / row.annotation).resolve()\n    if not image_path.is_relative_to(root) or not annotation_path.is_relative_to(root):\n        raise ValueError("Invalid image/annotation path")\n    reviewer = str(payload.get("reviewer", "")).strip()\n    if not reviewer or len(reviewer) > 90:\n        raise ValueError("A named reviewer is required")\n    category = str(payload.get("category", ""))\n    if category == "SKIP_UNCERTAIN":\n        if payload.get("boxes"):\n            raise ValueError("Undo all boxes before excluding an ambiguous tile")\n        manifest.loc[index, "review_status"] = "REJECTED"\n        manifest.loc[index, "reviewed_label"] = "UNKNOWN"\n        manifest.loc[index, "reviewer"] = reviewer\n        manifest.loc[index, "source_checked"] = "NO"\n        manifest.to_csv(csv_path, index=False)\n        return {"ok": True, "saved": "EXCLUDED: no training label created",\n                "remaining_unreviewed": int(manifest.review_status.eq("UNREVIEWED").sum())}\n    if payload.get("confirmed") is not True:\n        raise ValueError("You must confirm imagery rights, location and building function were reviewed")\n    if category not in ("HOSPITAL_PRESENT", "NO_HOSPITAL"):\n        raise ValueError("Explicit hospital present / no hospital choice required")\n    boxes = payload.get("boxes")\n    if not isinstance(boxes, list) or len(boxes) > 100:\n        raise ValueError("Invalid number of boxes")\n    if category == "HOSPITAL_PRESENT" and not boxes:\n        raise ValueError("Positive tile requires at least one drawn hospital roof rectangle")\n    if category == "NO_HOSPITAL" and boxes:\n        raise ValueError("Negative tile cannot contain hospital rectangles; undo boxes first")\n    with Image.open(image_path) as im:\n        width, height = im.size\n    shapes = []\n    for box in boxes:\n        if not isinstance(box, list) or len(box) != 4:\n            raise ValueError("Each drawn box requires four numeric coordinates")\n        x1, y1, x2, y2 = [float(v) for v in box]\n        if not (0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height and (x2-x1)>=3 and (y2-y1)>=3):\n            raise ValueError("Box corners must be inside the image with nonzero area")\n        shapes.append({"label": "hospital", "points": [[x1,y1],[x2,y2]],\n                       "group_id": None, "description": "Human-reviewed hospital rooftop", "shape_type": "rectangle", "flags": {}})\n    labelme = {"version": "5.5.0", "flags": {"verified": True}, "shapes": shapes,\n               "imagePath": image_path.name, "imageData": None, "imageHeight": height,\n               "imageWidth": width, "reviewed_by": reviewer,\n               "review_note": "Building use checked by a human. Data source and license verified."}\n    annotation_path.parent.mkdir(parents=True, exist_ok=True)\n    annotation_path.write_text(json.dumps(labelme, indent=2), encoding="utf-8")\n    manifest.loc[index, "review_status"] = "APPROVED"\n    manifest.loc[index, "reviewed_label"] = category\n    manifest.loc[index, "reviewer"] = reviewer\n    manifest.loc[index, "source_checked"] = "YES"\n    manifest.to_csv(csv_path, index=False)\n    remaining = int(manifest.review_status.eq("UNREVIEWED").sum())\n    return {"ok": True, "saved": str(annotation_path), "remaining_unreviewed": remaining}\n\n\n\ndef review_saved_status(root: str | Path, tile_id: str) -> dict[str, Any]:\n    """Read the real manifest and annotation from disk; do not trust a UI message."""\n    root = Path(root).resolve()\n    manifest = pd.read_csv(root / "review_manifest.csv", keep_default_na=False)\n    rows = manifest.loc[manifest.tile_id.astype(str).eq(str(tile_id))]\n    if len(rows) != 1:\n        return {"ok": False, "saved": False, "error": "Review tile not found"}\n    row = rows.iloc[0]\n    status = str(row.review_status)\n    path = (root / str(row.annotation)).resolve()\n    safe_path = path.is_relative_to(root)\n    saved = status == "REJECTED" or (status == "APPROVED" and safe_path and path.is_file())\n    return {"ok": True, "saved": bool(saved), "review_status": status,\n            "remaining_unreviewed": int(manifest.review_status.eq("UNREVIEWED").sum())}\n\n\ndef colab_save_review_result(root: str | Path, payload: dict[str, Any]) -> JSON:\n    """Return IPython.display.JSON, which Colab\'s invokeFunction actually transports.\n\n    Returning a plain Python dict is not enough for Colab\'s documented\n    response.data[\'application/json\'] contract.\n    """\n    try:\n        result = save_annotation_review(root, payload)\n        verified = review_saved_status(root, str(payload.get("tile_id", "")))\n        if not verified.get("saved"):\n            result = {"ok": False, "error": "Save returned but the annotation did not persist."}\n        else:\n            result["remaining_unreviewed"] = verified["remaining_unreviewed"]\n        return JSON(result)\n    except Exception as exc:\n        return JSON({"ok": False, "error": f"{type(exc).__name__}: {exc}"[:450]})\n\n\ndef colab_review_status_result(root: str | Path, tile_id: str) -> JSON:\n    """Independent read-back for an ambiguous or lost Colab browser response."""\n    try:\n        return JSON(review_saved_status(root, tile_id))\n    except Exception as exc:\n        return JSON({"ok": False, "saved": False,\n                     "error": f"{type(exc).__name__}: {exc}"[:450]})\n\ndef next_unreviewed_payload(root: str | Path) -> dict[str, Any]:\n    """Fetch the next *actual* pending image without rerunning a notebook cell.\n\n    Includes cached OSM proposals but never converts proposals into approval.\n    """\n    root = Path(root).resolve()\n    df = pd.read_csv(root / \'review_manifest.csv\', keep_default_na=False)\n    waiting = df.loc[df.review_status.eq(\'UNREVIEWED\')]\n    if waiting.empty:\n        return {\'ok\': True, \'done\': True, \'remaining\': 0}\n    row = waiting.iloc[0]\n    path = (root / str(row.image)).resolve()\n    if not path.is_relative_to(root) or not path.is_file():\n        raise FileNotFoundError(\'Next review image missing or outside project root\')\n    with Image.open(path) as im:\n        width, height = im.size\n        mime = Image.MIME.get(im.format, \'image/png\')\n    if width>1500 or height>1500 or mime not in (\'image/png\',\'image/jpeg\',\'image/webp\'):\n        raise ValueError(\'Unsupported annotation image\')\n    from osm_auto_annotations import load_draft_boxes\n    return {\'ok\':True, \'done\':False, \'remaining\':int(len(waiting)),\n            \'tile_id\':str(row.tile_id), \'name\':str(row[\'name\']),\n            \'candidate_class\':str(row.candidate_class), \'width\':int(width),\n            \'height\':int(height), \'image_src\':f\'data:{mime};base64,\'+\n            base64.b64encode(path.read_bytes()).decode(\'ascii\'),\n            \'suggested_boxes\':load_draft_boxes(root,str(row.tile_id)),\n            \'draft_status\':\'OPENSTREETMAP DRAFT — verify roof and hospital use\'}\n\n\ndef colab_next_review_result(root: str | Path) -> JSON:\n    try:\n        return JSON(next_unreviewed_payload(root))\n    except Exception as exc:\n        return JSON({\'ok\':False,\'error\':f\'{type(exc).__name__}: {exc}\'[:400]})\n\n\ndef build_labeler_ui(tile_id: str, name: str, candidate_class: str, image_path: Path, suggested_boxes=None):\n    """Return static HTML and executable JavaScript as separate notebook outputs.\n\n    Colab may render HTML while not executing embedded <script> elements.\n    The image is therefore a normal <img> element, visible even when JavaScript\n    fails, and we execute event bindings explicitly with display(Javascript).\n    """\n    image_path = Path(image_path)\n    with Image.open(image_path) as src:\n        width, height = src.size\n        mime = Image.MIME.get(src.format, "image/png")\n    if width > 1500 or height > 1500:\n        raise ValueError("Review UI supports images up to 1500 pixels per side")\n    if mime not in ("image/png", "image/jpeg", "image/webp"):\n        raise ValueError("Review UI supports PNG/JPEG/WebP imagery")\n\n    root_id = "oss4008-review-" + uuid.uuid4().hex[:12]\n    img_uri = f"data:{mime};base64,{base64.b64encode(image_path.read_bytes()).decode(\'ascii\')}"\n    title = html.escape(f"{name} · {candidate_class} · {tile_id}", quote=True)\n    # Safe static HTML: never depend on a script to display the aerial photograph.\n    markup = f\'\'\'<div id="{root_id}" style="background:#f6f8fb;color:#203449;padding:14px;border:1px solid #c8d2df;border-radius:10px;font:14px system-ui;max-width:950px">\n      <div style="font-weight:750;font-size:16px">Hospital AI · Manual roof review</div>\n      <p data-role="title" style="color:#435a76;font-weight:600">{title}</p>\n      <p style="color:#435a76">The image below is real overhead imagery. Outline only roofs of buildings that you have independently confirmed are hospitals. A reference location alone is not proof.</p>\n      <div style="position:relative;width:100%;max-width:640px;border:1px solid #b9c7dc;background:#e9eef4;line-height:0">\n        <img data-role="image" src="{img_uri}" width="{width}" height="{height}" alt="Aerial photograph to label" style="width:100%;height:auto;display:block;"/>\n        <canvas data-role="canvas" width="{width}" height="{height}" style="position:absolute;inset:0;width:100%;height:100%;cursor:crosshair;touch-action:none"></canvas>\n      </div>\n      <p><button data-role="undo" type="button">Undo last box</button> <button data-role="use-drafts" type="button">Use suggested OSM boxes</button> <span data-role="count">0 rooftop rectangles</span> <span data-role="draft-count" style="color:#2563eb">Drafts are unverified</span></p>\n      <div style="display:flex;flex-wrap:wrap;gap:12px;align-items:center">\n        <label style="color:#203449">Verified classification\n          <select data-role="category" style="color:#203449;background:white;border:1px solid #9aaac0;padding:5px">\n            <option value="">Select...</option>\n            <option value="HOSPITAL_PRESENT">Hospital rooftop(s) drawn</option>\n            <option value="NO_HOSPITAL">No hospital rooftop in this image</option>\n            <option value="SKIP_UNCERTAIN">Cannot verify: exclude image</option>\n          </select>\n        </label>\n        <label style="color:#203449">Reviewer <input data-role="reviewer" placeholder="Your initials/name" style="color:#203449;background:white;border:1px solid #9aaac0;padding:5px"/></label>\n      </div>\n      <p><label style="color:#203449"><input data-role="confirm" type="checkbox"/> I personally checked the building function, image source and permission to use the image for training.</label></p>\n      <button data-role="save" type="button" style="background:#235bdd;color:white;padding:10px;border:0;border-radius:6px">Save reviewed annotation</button>\n      <p data-role="status" aria-live="polite" style="color:#314860;white-space:pre-wrap">Image displayed. Initializing drawing controls…</p>\n    </div>\'\'\'\n\n    config = json.dumps({"root_id": root_id, "tile_id": str(tile_id),\n                         "width": width, "height": height, "drafts": suggested_boxes or []}, ensure_ascii=False)\n    # Put JS in an actual Javascript MIME output rather than static HTML.\n    # No untrusted filename or user\'s free text is interpolated as JS source.\n    js = r\'\'\'(() => {\n      const config = __CONFIG__;\n      const root = document.getElementById(config.root_id);\n      if (!root) throw new Error(\'Annotation HTML not ready in Colab output frame\');\n      const get = role => root.querySelector(\'[data-role="\' + role + \'"]\');\n      const canvas = get(\'canvas\');\n      const img = get(\'image\');\n      const status = get(\'status\');\n      const ctx = canvas.getContext(\'2d\');\n      if (!ctx) { status.textContent = \'Drawing unavailable: browser 2D canvas unsupported.\'; return; }\n      const boxes = [];\n      const drafts = [...config.drafts];\n      let start = null, cursor = null;\n      const coords = event => {\n        const r = canvas.getBoundingClientRect();\n        return [Math.max(0, Math.min(config.width, (event.clientX-r.left)*config.width/r.width)),\n                Math.max(0, Math.min(config.height,(event.clientY-r.top)*config.height/r.height))];\n      };\n      const paint = () => {\n        ctx.clearRect(0, 0, config.width, config.height);\n        ctx.lineWidth = 3;\n        ctx.strokeStyle = \'#ffe000\';\n        for (const b of boxes) ctx.strokeRect(b[0], b[1], b[2]-b[0], b[3]-b[1]);\n        ctx.save();ctx.setLineDash([8,5]);ctx.strokeStyle=\'#2d70ff\';\n        for (const b of drafts) ctx.strokeRect(b[0],b[1],b[2]-b[0],b[3]-b[1]);\n        ctx.restore();\n        get(\'draft-count\').textContent=drafts.length+\' OSM draft boxes (BLUE DASHED = unverified)\';\n        if (start && cursor) {\n          const b = [Math.min(start[0],cursor[0]),Math.min(start[1],cursor[1]),\n                     Math.max(start[0],cursor[0]),Math.max(start[1],cursor[1])];\n          ctx.strokeStyle = \'#2ce2b8\';\n          ctx.strokeRect(b[0], b[1], b[2]-b[0], b[3]-b[1]);\n        }\n        get(\'count\').textContent = boxes.length + \' rooftop rectangles\';\n      };\n      canvas.addEventListener(\'pointerdown\', event => {\n        if (get(\'save\').disabled) return;\n        canvas.setPointerCapture(event.pointerId);\n        start = coords(event); cursor = start; paint();\n      });\n      canvas.addEventListener(\'pointermove\', event => {\n        if (start) { cursor = coords(event); paint(); }\n      });\n      canvas.addEventListener(\'pointerup\', event => {\n        if (!start) return;\n        cursor = coords(event);\n        const b = [Math.min(start[0],cursor[0]),Math.min(start[1],cursor[1]),\n                   Math.max(start[0],cursor[0]),Math.max(start[1],cursor[1])];\n        if (b[2]-b[0]>=3 && b[3]-b[1]>=3) boxes.push(b);\n        start = null; cursor = null; paint();\n      });\n      canvas.addEventListener(\'pointercancel\', () => { start=null; cursor=null; paint(); });\n      get(\'undo\').onclick = () => { boxes.pop(); paint(); };\n      get(\'use-drafts\').onclick = () => {\n        if(!drafts.length){status.textContent=\'No mapped building footprint proposals for this image. Draw manually or exclude uncertain image.\';return;}\n        boxes.push(...drafts.map(b=>b.slice()));drafts.length=0;paint();\n        status.textContent=\'OSM draft outlines copied. CHECK EACH BOX and remove incorrect boxes (Undo) before approving. OSM geometry is NOT proof of hospital use.\';\n      };\n      const readJson = reply => {\n        let value=reply?.data?.[\'application/json\'];\n        if (typeof value===\'string\'){try{value=JSON.parse(value);}catch(_){}}\n        if(Array.isArray(value)&&value.length===1)value=value[0];\n        return value;\n      };\n      async function openNext(api){\n        status.textContent=\'Saved to Google Drive. Loading next aerial image…\';\n        try {\n          const payload=readJson(await api.invokeFunction(\'oss4008_hospital_ai.next_review\',[],{}));\n          if(!payload?.ok){status.textContent=\'Saved, but next image unavailable: \'+String(payload?.error||\'unknown error\');return;}\n          if(payload.done){\n            status.textContent=\'ALL REVIEWS FINISHED. Rerun the training cell to check training requirements.\';\n            get(\'save\').disabled=true;get(\'use-drafts\').disabled=true;return;\n          }\n          config.tile_id=payload.tile_id; config.width=payload.width;config.height=payload.height;\n          img.src=payload.image_src;img.width=payload.width;img.height=payload.height;\n          canvas.width=payload.width;canvas.height=payload.height;\n          get(\'title\').textContent=payload.name+\' · \'+payload.candidate_class+\' · \'+payload.tile_id;\n          boxes.length=0;drafts.length=0;\n          drafts.push(...(payload.suggested_boxes||[]).map(b=>b.slice()));\n          get(\'category\').value=\'\';get(\'confirm\').checked=false;\n          start=null;cursor=null;get(\'save\').disabled=false;paint();\n          status.textContent=\'NEXT IMAGE OPENED automatically. \'+payload.remaining+\' remain. Blue dashed OSM proposals require verification.\';\n        }catch(error){status.textContent=\'Saved, but could not advance automatically: \'+String(error).slice(0,160);}\n      }\n      get(\'save\').onclick = async () => {\n        const button = get(\'save\');\n        if (!get(\'reviewer\').value.trim()) { status.textContent = \'Enter reviewer name or initials first.\'; return; }\n        const category = get(\'category\').value;\n        if (!category) { status.textContent = \'Select hospital, no hospital, or exclude this image.\'; return; }\n        if (category === \'HOSPITAL_PRESENT\' && boxes.length===0) {\n          status.textContent = \'Draw at least one rectangle around a verified hospital roof.\'; return;\n        }\n        if (category !== \'HOSPITAL_PRESENT\' && boxes.length>0) {\n          status.textContent = \'Undo rectangles before choosing no hospital or exclude.\'; return;\n        }\n        if (category !== \'SKIP_UNCERTAIN\' && !get(\'confirm\').checked) {\n          status.textContent = \'Confirm the building use, imagery source and training rights before saving.\'; return;\n        }\n        const api = window.google?.colab?.kernel;\n        if (!api || typeof api.invokeFunction !== \'function\') {\n          status.textContent = \'Colab callback unavailable. Rerun the annotation cell in an active Colab runtime.\';\n          return;\n        }\n        button.disabled = true;\n        status.textContent = \'Saving review to Google Drive…\';\n        const payload = {tile_id:config.tile_id, category,\n          reviewer:get(\'reviewer\').value.trim(), confirmed:get(\'confirm\').checked,\n          boxes:boxes.map(box=>box.slice())};\n        try {\n          // Google Colab\'s official callback format is JSON data in application/json.\n          const response = await api.invokeFunction(\'oss4008_hospital_ai.save_review\',[payload],{});\n          const result = readJson(response);\n          if (result?.ok) {\n            await openNext(api);\n          } else if (result?.error) {\n            status.textContent = \'Could not save: \' + result.error;\n            button.disabled = false;\n          } else {\n            // Older Colab outputs can return an empty browser message even\n            // after Python finished. Verify the actual files before reporting failure.\n            const verifyResponse = await api.invokeFunction(\'oss4008_hospital_ai.review_status\',[config.tile_id],{});\n            const verified = readJson(verifyResponse);\n            if (verified?.saved) {\n              await openNext(api);\n            } else {\n              status.textContent = \'Save not confirmed: \' + (verified?.error || \'No reviewed annotation found on disk.\');\n              button.disabled = false;\n            }\n          }\n        } catch (error) {\n          status.textContent = \'Could not save: \' + String(error);\n          button.disabled = false;\n        }\n      };\n      status.textContent = \'Ready. Drag across a verified hospital roof to draw a rectangle.\';\n      paint();\n    })();\'\'\'.replace(\'__CONFIG__\', config)\n    return markup, js\n\n\ndef launch_labeler(root: str | Path) -> dict[str, Any]:\n    """Show one image in Colab and bind its drawing controls with Javascript MIME.\n\n    All annotation data remains in Drive. Nothing is auto-approved or labeled.\n    """\n    root = Path(root).resolve()\n    manifest = pd.read_csv(root / "review_manifest.csv", keep_default_na=False)\n    pending = manifest.loc[manifest.review_status.eq("UNREVIEWED")]\n    if pending.empty:\n        print("No unreviewed images remain. Check approved/omitted site coverage before training.")\n        return {"remaining": 0}\n    row = pending.iloc[0]\n    image_path = (root / str(row.image)).resolve()\n    if not image_path.is_relative_to(root) or not image_path.is_file():\n        raise FileNotFoundError(f"Missing candidate image: {row.image}")\n    from google.colab import output\n    output.register_callback("oss4008_hospital_ai.save_review",\n                             lambda payload: colab_save_review_result(root, payload))\n    output.register_callback("oss4008_hospital_ai.review_status",\n                             lambda tile_id: colab_review_status_result(root, tile_id))\n    output.register_callback("oss4008_hospital_ai.next_review",\n                             lambda: colab_next_review_result(root))\n    from osm_auto_annotations import load_draft_boxes\n    suggested = load_draft_boxes(root,str(row.tile_id))\n    markup, script = build_labeler_ui(str(row.tile_id), str(row["name"]),\n                                     str(row.candidate_class), image_path, suggested_boxes=suggested)\n    display(HTML(markup))\n    display(Javascript(script))\n    return {"tile_id": str(row.tile_id), "remaining": len(pending),\n            "human_review_required": True}\n'
OSM_AUTO_SOURCE = '"""OpenStreetMap-assisted *DRAFT* roof boxes for the OSS4008 hospital project.\n\nOSM describes mapped buildings, not verified current hospital use. Every draft\nremains outside the ground-truth LabelMe folder and is excluded from training.\nNo data here is a trained-model prediction or a validated hospital label.\n"""\nfrom __future__ import annotations\n\nimport base64\nfrom datetime import datetime, timezone\nfrom io import BytesIO\nimport json\nimport math\nfrom pathlib import Path\nimport time\nfrom typing import Any\n\nimport pandas as pd\nimport requests\nimport rasterio\nfrom pyproj import Transformer\nfrom shapely.geometry import Polygon, Point\nfrom shapely.ops import transform as geometry_transform\nfrom PIL import Image, ImageDraw, ImageFont\n\nOVERPASS_ENDPOINT = \'https://overpass.private.coffee/api/interpreter\'\n# Public mirrors are not guaranteed. Use only after a connection/service failure,\n# never to circumvent an explicit 429 quota/rate-limit response.\nOVERPASS_FALLBACK_ENDPOINTS = (\'https://overpass-api.de/api/interpreter\',)\nOSM_LICENSE_URL = \'https://www.openstreetmap.org/copyright\'\nOSM_CREDIT = \'© OpenStreetMap contributors (ODbL)\'\nBLOCKED_BUILDING_TYPES = {\'no\',\'roof\',\'garage\',\'garages\',\'carport\',\'shed\',\'house\',\'detached\',\n                          \'residential\',\'apartments\',\'semidetached_house\',\'bungalow\',\'terrace\',\n                          \'hut\',\'greenhouse\',\'parking\',\'industrial\',\'warehouse\'}\n\n\ndef query_for_centers(centers: list[tuple[float,float]], radius_m: int = 180) -> str:\n    if not centers or len(centers)>8 or not 70<=radius_m<=300:\n        raise ValueError(\'Query must include 1..8 centres and a 70..300 m radius\')\n    parts=[]\n    for lat,lon in centers:\n        if not 41<=float(lat)<=56 or not -96<=float(lon)<=-70:\n            raise ValueError(\'Only Canadian Ontario-region coordinates are supported by the pilot\')\n        position=f\'(around:{radius_m},{lat:.6f},{lon:.6f})\'\n        for tag in (\'["building"]\',\'["amenity"="hospital"]\',\'["healthcare"="hospital"]\'):\n            parts.append(\'way\'+tag+position+\';\')\n        parts.extend([\'relation["amenity"="hospital"]\'+position+\';\',\n                      \'relation["healthcare"="hospital"]\'+position+\';\'])\n    return \'[out:json][timeout:20];\\n(\\n\'+ \'\\n\'.join(parts)+\'\\n);\\nout geom;\'\n\n\ndef fetch_overpass(query: str, session: Any | None = None, endpoint: str = OVERPASS_ENDPOINT,\n                   retries: int = 0, sleep=time.sleep,\n                   fallback_endpoints: tuple[str, ...] | None = None) -> dict[str, Any]:\n    """Sequential fallback for timeouts/5xx; never evade OSM HTTP 429.\n\n    Use the provider\'s current private.coffee hostname. Only try the public\n    secondary instance when the primary cannot answer. Callers supply a session\n    or fetcher in synthetic tests, so real live reliability is not presumed.\n    """\n    if not query or len(query) > 20000 or not 0 <= retries <= 2:\n        raise ValueError(\'Invalid Overpass query/retry configuration\')\n    if fallback_endpoints is None:\n        # Injected mock sessions default to one endpoint; explicit tests can\n        # still supply fallback_endpoints to exercise sequential failover.\n        fallback_endpoints = OVERPASS_FALLBACK_ENDPOINTS if session is None else ()\n    servers = (endpoint,) + tuple(x for x in fallback_endpoints if x != endpoint)\n    if not all(s.startswith(\'https://\') and \'/api/interpreter\' in s for s in servers):\n        raise ValueError(\'Invalid HTTPS Overpass server\')\n    http = session if session is not None else requests.Session()\n    last_error = None\n    headers = {\'User-Agent\':\'OSS4008-HospitalInfrastructureCourse/1.0 (non-commercial education)\',\n               \'Accept\':\'application/json\'}\n    for idx, server in enumerate(servers):\n        for trial in range(retries + 1):\n            try:\n                response = http.post(server, data={\'data\': query},\n                                     timeout=(5, 20), headers=headers)\n                if response.status_code == 429:\n                    # An explicit rate limit must never be bypassed by rotating\n                    # to another public server in this teaching workflow.\n                    raise RuntimeError(\'Overpass returned 429: rate limit; stop and retry later\')\n                if response.status_code in (502, 503, 504):\n                    raise requests.HTTPError(f\'Overpass returned HTTP {response.status_code}\')\n                response.raise_for_status()\n                if hasattr(response, \'content\') and len(response.content) > 25_000_000:\n                    raise ValueError(\'Unexpectedly large OSM response; narrow the query\')\n                result = response.json()\n                if not isinstance(result, dict) or not isinstance(result.get(\'elements\'), list):\n                    raise ValueError(\'Overpass returned no usable element list\')\n                if len(result[\'elements\']) > 25000:\n                    raise ValueError(\'OSM response too large; use smaller batches\')\n                return result\n            except RuntimeError as exc:\n                # Do not evade a provider rate limit.\n                if \'429\' in str(exc):\n                    raise\n                last_error = exc\n            except (requests.RequestException, ValueError) as exc:\n                last_error = exc\n            if trial < retries:\n                sleep(min(6, 2 * (trial + 1)))\n        if idx + 1 < len(servers):\n            print(\'OSM primary unavailable; trying separate public Overpass mirror...\', flush=True)\n    raise RuntimeError(f\'OpenStreetMap fetch failed after {len(servers)} server(s): {last_error}\')\n\n\ndef _way_polygons(element: dict) -> list[Polygon]:\n    """Get WGS84 polygons, including relation *outer* member boundaries."""\n    raw=[]\n    if element.get(\'type\')==\'way\':\n        raw=[element.get(\'geometry\') or []]\n    elif element.get(\'type\')==\'relation\':\n        raw=[v.get(\'geometry\') or [] for v in element.get(\'members\',[])\n             if v.get(\'type\')==\'way\' and v.get(\'role\')==\'outer\']\n    result=[]\n    for ring in raw:\n        try:\n            coords=[(float(p[\'lon\']),float(p[\'lat\'])) for p in ring]\n            if len(coords)<4 or coords[0]!=coords[-1]:\n                continue\n            poly=Polygon(coords)\n            if not poly.is_valid:\n                poly=poly.buffer(0)\n            if poly.geom_type==\'Polygon\' and not poly.is_empty and poly.area>0:\n                result.append(poly)\n        except (ValueError,TypeError,KeyError):\n            pass\n    return result\n\n\ndef extract_suggestions(elements: list[dict], lat: float, lon: float,\n                        geotiff: str | Path, max_boxes: int = 7) -> list[dict]:\n    """Find candidate *roof* footprints, NEVER hospital-site boundary shapes.\n\n    Confidence labels mean evidence strength for HUMAN review, not model\n    probabilities. Generic buildings are only considered if inside a mapped\n    hospital campus or very near a known-site point; neither proves hospital use.\n    """\n    if not 0<=max_boxes<=25:\n        raise ValueError(\'max_boxes must be 0..25\')\n    grounds=[]\n    buildings=[]\n    for element in elements:\n        tags=element.get(\'tags\') or {}\n        is_hospital=tags.get(\'amenity\')==\'hospital\' or tags.get(\'healthcare\')==\'hospital\'\n        btype=tags.get(\'building\',\'\')\n        polys=_way_polygons(element)\n        if is_hospital and (not btype or btype==\'no\'):\n            grounds.extend(polys)\n        if btype and btype not in BLOCKED_BUILDING_TYPES:\n            for poly in polys:\n                buildings.append((element,poly))\n    # Clip in ORIGINAL georeferenced image space. A 100m road/campus polygon\n    # cannot be treated as a building roof merely because it contains the site.\n    candidates=[]\n    with rasterio.open(geotiff) as ds:\n        if ds.crs is None or ds.width<50 or ds.height<50:\n            raise ValueError(\'Image lacks valid geographic reference\')\n        transformer=Transformer.from_crs(\'EPSG:4326\',ds.crs,always_xy=True)\n        px_inverse=~ds.transform\n        ref_geom=Point(*transformer.transform(lon,lat))\n        # Point-distance uses projected metres; Ontario imagery stores EPSG:3857.\n        for element,poly in buildings:\n            tags=element.get(\'tags\') or {}\n            try:\n                proj=geometry_transform(transformer.transform,poly)\n                if proj.is_empty or proj.area<35 or proj.area>60000:\n                    continue\n                geo_centre=proj.centroid\n                # Web Mercator overstates ground distance by ~1/cos(lat).\n                meters=geo_centre.distance(ref_geom)*max(.25,math.cos(math.radians(lat)))\n                matched=tags.get(\'amenity\')==\'hospital\' or tags.get(\'healthcare\')==\'hospital\'\n                typed=tags.get(\'building\')==\'hospital\'\n                campus=any(proj.intersection(geometry_transform(transformer.transform,g)).area/\n                           max(proj.area,1)>0.55 for g in grounds)\n                if matched:\n                    why=\'explicit_hospital_use_tag\'\n                    strength=3\n                elif typed:\n                    why=\'building_hospital_tag_not_confirmed_current_use\'\n                    strength=2\n                elif campus:\n                    why=\'building_inside_mapped_hospital_campus\'\n                    strength=1\n                elif meters <= 95 and not grounds:\n                    why=\'near_odhf_reference_only\'\n                    strength=0\n                else:\n                    continue\n                # Pixel affine inverse: x=column; y=row.\n                coords=[px_inverse * p for p in proj.exterior.coords]\n                x1=max(0.0,min(p[0] for p in coords));x2=min(float(ds.width),max(p[0] for p in coords))\n                y1=max(0.0,min(p[1] for p in coords));y2=min(float(ds.height),max(p[1] for p in coords))\n                if x2-x1 < 6 or y2-y1 < 6 or (x2-x1)*(y2-y1)>ds.width*ds.height*.75:\n                    continue\n                raw=[p[0] for p in coords]+[p[1] for p in coords]\n                if not all(math.isfinite(v) for v in raw):continue\n                candidates.append({\n                    \'osm_id\':f"{element[\'type\']}/{element[\'id\']}",\'bbox\':[round(x1,1),round(y1,1),round(x2,1),round(y2,1)],\n                    \'reason\':why,\'evidence_level\':strength,\'point_distance_m\':round(meters,1),\n                    \'osm_tags\':{k:str(v)[:120] for k,v in tags.items() if k in (\'building\',\'amenity\',\'healthcare\',\'name\')},\n                })\n            except (ValueError,TypeError,OverflowError):\n                continue\n    # Strong OSM tags first; then closest campus/point buildings. Limit weak\n    # near-reference guesses to at most three per tile, never silently mark as hospital.\n    candidates.sort(key=lambda x:(-x[\'evidence_level\'],x[\'point_distance_m\']))\n    accepted=[]; weak_count=0\n    def iou(a,b):\n        l=max(a[0],b[0]);t=max(a[1],b[1]);r=min(a[2],b[2]);bt=min(a[3],b[3]);inter=max(0,r-l)*max(0,bt-t)\n        return inter/max(1,(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter)\n    for item in candidates:\n        if item[\'evidence_level\']==0 and weak_count>=3:continue\n        if any(iou(item[\'bbox\'],other[\'bbox\'])>.70 for other in accepted):continue\n        accepted.append(item)\n        if item[\'evidence_level\']==0:weak_count+=1\n        if len(accepted)>=max_boxes:break\n    return accepted\n\n\ndef load_draft_boxes(root: str | Path,tile_id: str) -> list[list[float]]:\n    """Read local proposed boxes; these never indicate approved hospital use."""\n    root=Path(root).resolve()\n    fname=root/\'drafts\'/\'osm\'/f\'{tile_id}.json\'\n    if not fname.is_file():return []\n    doc=json.loads(fname.read_text(encoding=\'utf-8\'))\n    if doc.get(\'tile_id\')!=tile_id or doc.get(\'approved\') is not False:\n        raise ValueError(\'OSM draft identity/status mismatch\')\n    return [r[\'bbox\'] for r in doc.get(\'candidates\',[]) if len(r.get(\'bbox\',[]))==4]\n\n\ndef generate_osm_drafts(root: str | Path, limit: int = 50, batch_size: int = 4,\n                        session: Any | None = None, pause_s: float = 1.5,\n                        sleeper=time.sleep, fetcher=fetch_overpass,\n                        overwrite_drafts: bool = False) -> dict[str, Any]:\n    """Batch Overpass lookups, cache, create draft boxes (NEVER approved labels).\n\n    Returns actual coverage; \'100\' is a target, not a promise. Skips already\n    approved/rejected images; never touches review_manifest.csv or verified JSON.\n    """\n    root=Path(root).resolve()\n    if not 1<=limit<=200 or not 1<=batch_size<=8:\n        raise ValueError(\'Unsupported draft target/batch size\')\n    manifest_path=root/\'review_manifest.csv\'\n    if not manifest_path.is_file():\n        raise FileNotFoundError(\'Download pilot aerial images first: review_manifest.csv missing\')\n    df=pd.read_csv(manifest_path,keep_default_na=False)\n    if not {\'tile_id\',\'image\',\'geotiff\',\'latitude\',\'longitude\',\'candidate_class\',\'review_status\'}.issubset(df.columns):\n        raise ValueError(\'Manifest missing required imagery/label fields\')\n    df=df.loc[df.candidate_class.eq(\'known_hospital_site\') & df.review_status.eq(\'UNREVIEWED\')].head(limit)\n    rows=[]; errors=[]; pending=[]; skipped=0\n    outdir=root/\'drafts\'/\'osm\';outdir.mkdir(parents=True,exist_ok=True)\n    for _,r in df.iterrows():\n        name=str(r.tile_id)\n        if \'/\' in name or \'..\' in name or not name:\n            errors.append(f\'{name}: invalid tile id\');continue\n        if not (root/str(r.image)).is_file() or not (root/str(r.geotiff)).is_file():\n            errors.append(f\'{name}: missing image or GeoTIFF\');continue\n        path=outdir/f\'{name}.json\'\n        if path.is_file() and not overwrite_drafts:\n            try:\n                prev=json.loads(path.read_text(encoding=\'utf-8\'))\n                if prev.get(\'approved\') is False:\n                    rows.append({\'tile_id\':name,\'draft_boxes\':len(prev.get(\'candidates\',[])),\'cached\':True})\n                    skipped+=1;continue\n            except (OSError,ValueError):pass\n        try:\n            coords=(float(r.latitude),float(r.longitude))\n            if not (41<=coords[0]<=56 and -96<=coords[1]<=-70):\n                raise ValueError(\'Outside supported Ontario pilot region\')\n            pending.append((r,coords))\n        except ValueError as exc:\n            errors.append(f\'{name}: {exc}\')\n    # Show progress BEFORE contacting the remote server; interruption retains\n    # all existing images and completed per-image draft JSON files in Drive.\n    total_batches = math.ceil(len(pending) / batch_size)\n    print(f\'OSM matching: {len(df)} candidate hospital images (max {limit}), \'\n          f\'{skipped} cached, {len(pending)} to query in {total_batches} small batches.\', flush=True)\n    progress_file = root / \'drafts\' / \'osm_progress.json\'\n    def write_progress(stage, batch_number, last_message=\'\'):\n        progress = {\'stage\': stage, \'batch\': batch_number, \'total_batches\': total_batches,\n                    \'images_checked_or_cached\': len(rows),\n                    \'images_with_drafts\': sum(bool(item[\'draft_boxes\']) for item in rows),\n                    \'draft_boxes\': sum(item[\'draft_boxes\'] for item in rows),\n                    \'errors_so_far\': len(errors), \'last_message\': str(last_message)[:250],\n                    \'note\': \'Unverified OSM draft boxes, never auto-approved training labels\'}\n        tmp=progress_file.with_suffix(\'.json.tmp\')\n        tmp.write_text(json.dumps(progress, indent=2), encoding=\'utf-8\')\n        tmp.replace(progress_file)\n    write_progress(\'STARTED\', 0)\n    consecutive_failures=0\n    for start in range(0,len(pending),batch_size):\n        batch=pending[start:start+batch_size]\n        batch_number=start//batch_size+1\n        print(f\'OSM batch {batch_number}/{total_batches}: looking up {len(batch)} saved images...\', flush=True)\n        write_progress(\'REQUESTING\', batch_number)\n        query=query_for_centers([c for _,c in batch])\n        try:\n            response=fetcher(query,session=session,sleep=sleeper)\n            elements=response[\'elements\']\n        except Exception as exc:\n            consecutive_failures+=1\n            message=f\'OSM request {batch_number}: {type(exc).__name__}: {str(exc)[:160]}\'\n            errors.append(message)\n            print(f\'OSM batch {batch_number}/{total_batches} failed: {type(exc).__name__}: {str(exc)[:120]}\', flush=True)\n            write_progress(\'BATCH_FAILED\', batch_number, message)\n            # Do not hammer public OSM servers after repeated timeouts/429s.\n            if consecutive_failures>=2:\n                errors.append(\'Stopped after consecutive Overpass errors; existing draft files are preserved.\')\n                print(\'OSM: stopping after two failed batches. Previously saved drafts are intact.\', flush=True)\n                write_progress(\'STOPPED_ON_SERVER_ERRORS\', batch_number)\n                break\n            continue\n        consecutive_failures=0\n        for r,(lat,lon) in batch:\n            try:\n                suggested=extract_suggestions(elements,lat,lon,root/str(r.geotiff))\n                data={\'tile_id\':str(r.tile_id),\'approved\':False,\'verification\':\'REQUIRES_HUMAN_REVIEW\',\n                      \'source\':\'OpenStreetMap\', \'license_url\':OSM_LICENSE_URL,\'attribution\':OSM_CREDIT,\n                      \'created_at_utc\':datetime.now(timezone.utc).isoformat(),\'candidate_site\':str(r.get(\'name\',\'\')),\n                      \'candidate_class\':str(r.candidate_class),\'candidates\':suggested}\n                target=outdir/f\'{r.tile_id}.json\';temp=target.with_suffix(\'.json.tmp\')\n                temp.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n                temp.replace(target)\n                rows.append({\'tile_id\':str(r.tile_id),\'draft_boxes\':len(suggested),\'cached\':False})\n            except Exception as exc:\n                errors.append(f\'{r.tile_id}: {type(exc).__name__}: {str(exc)[:130]}\')\n        print(f\'OSM batch {batch_number}/{total_batches} complete: \'\n              f\'{len(rows)} images checked (including cached); \'\n              f\'{sum(bool(item["draft_boxes"]) for item in rows)} images have draft boxes.\', flush=True)\n        write_progress(\'BATCH_COMPLETE\', batch_number)\n        if start+batch_size<len(pending) and pause_s>0:\n            sleeper(pause_s)\n    summary={\'requested_max\':limit,\'candidate_images\':len(df),\'queried_or_cached\':len(rows),\n             \'images_with_drafts\':sum(bool(r[\'draft_boxes\']) for r in rows),\n             \'proposed_boxes\':sum(r[\'draft_boxes\'] for r in rows),\'already_cached\':skipped,\n             \'approved_automatically\':0,\'trained_models\':0,\n             \'errors\':errors[:50],\'attribution\':OSM_CREDIT,\n             \'message\':\'OSM polygons are draft boxes ONLY. Independently verify hospital function and alignment before training.\'}\n    report=root/\'drafts\'/\'osm_inventory.json\'\n    report.write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n    pd.DataFrame(rows,columns=[\'tile_id\',\'draft_boxes\',\'cached\']).to_csv(root/\'drafts\'/\'osm_inventory.csv\',index=False)\n    write_progress(\'FINISHED\', total_batches)\n    print(f\'OSM matching finished: {summary["images_with_drafts"]} images with drafts, \'\n          f\'{summary["proposed_boxes"]} draft boxes; automatically approved: 0.\', flush=True)\n    return summary\n\n\ndef make_draft_contact_sheet(root: str | Path, max_items: int = 36, thumb: int = 230) -> Path:\n    """Prepare a compact visual review grid with tentative proposed boxes."""\n    root=Path(root)\n    df=pd.read_csv(root/\'review_manifest.csv\',keep_default_na=False)\n    tiles=[]\n    for _,r in df.iterrows():\n        draft=root/\'drafts\'/\'osm\'/f\'{r.tile_id}.json\'\n        image=root/str(r.image)\n        if draft.is_file() and image.is_file():\n            doc=json.loads(draft.read_text());\n            if doc.get(\'candidates\'):tiles.append((r,image,doc))\n            if len(tiles)>=max_items:break\n    if not tiles:\n        raise ValueError(\'No OSM draft boxes available to preview\')\n    cols=4;rows=math.ceil(len(tiles)/cols)\n    size=(cols*(thumb+12),rows*(thumb+48))\n    result=Image.new(\'RGB\',size,\'white\'); draw=ImageDraw.Draw(result)\n    for index,(r,image,doc) in enumerate(tiles):\n        with Image.open(image) as im:\n            w,h=im.size;resized=im.convert(\'RGB\').resize((thumb,thumb))\n        x=index%cols*(thumb+12);y=index//cols*(thumb+48)\n        result.paste(resized,(x,y))\n        for candidate in doc[\'candidates\']:\n            x1,y1,x2,y2=candidate[\'bbox\'];scx=thumb/w;scy=thumb/h\n            color=\'#2764ef\' if candidate[\'evidence_level\']>=2 else \'#efa629\'\n            draw.rectangle((x+x1*scx,y+y1*scy,x+x2*scx,y+y2*scy),outline=color,width=3)\n        draw.text((x+2,y+thumb+3),str(r[\'name\'])[:30],fill=\'#17213b\')\n        draw.text((x+2,y+thumb+21),\'DRAFT boxes: \'+str(len(doc[\'candidates\'])),fill=\'#9b5311\')\n    out=root/\'drafts\'/\'osm_contact_sheet.jpg\';out.parent.mkdir(parents=True,exist_ok=True)\n    result.save(out,quality=87)\n    return out\n'
OVERTURE_WEAK_SOURCE = '"""Experimental Overture building footprints near known ODHF hospital points.\n\nNO annotated building here is verified to be a hospital. This module proposes\nweak *training hypotheses*, does not modify any human-reviewed labels, and\nnever claims validation or hospital identity from proximity alone.\n"""\nfrom __future__ import annotations\n\nfrom datetime import datetime, timezone\nimport json\nimport math\nfrom pathlib import Path\nimport warnings\nfrom typing import Callable\n\nimport pandas as pd\nimport rasterio\nfrom rasterio.warp import transform_bounds\nfrom pyproj import Transformer\nfrom shapely.geometry import Point, Polygon, MultiPolygon, shape\nfrom shapely import wkb\nfrom shapely.ops import transform as geom_transform\n\nATTRIBUTION = \'© OpenStreetMap contributors, Overture Maps Foundation\'\nLICENSE_URL = \'https://docs.overturemaps.org/attribution/\'\nEXCLUDE_CLASSES = {\'house\', \'apartments\', \'residential\', \'garage\', \'garages\',\n                   \'shed\', \'carport\', \'school\', \'hotel\', \'university\', \'industrial\',\n                   \'warehouse\', \'retail\', \'commercial\', \'detached\', \'bungalow\'}\n\n\ndef fetch_overture_bbox(bbox):\n    """Fetch only buildings in a small bounding box with official Overture client.\n\n    Raise on missing STAC index rather than silently scanning the whole planet.\n    """\n    from overturemaps import record_batch_reader\n    from overturemaps.core import StacIndexWarning\n    with warnings.catch_warnings():\n        warnings.simplefilter(\'error\', StacIndexWarning)\n        reader = record_batch_reader(\'building\', bbox=bbox)\n        if reader is None:\n            return []\n        rows = reader.read_all()\n    if rows.num_rows > 6000:\n        raise RuntimeError(\'Too many Overture buildings for one roof chip; refusing large extraction\')\n    return rows.to_pylist()\n\n\ndef _geometry(item):\n    geom = item.get(\'geometry\')\n    if isinstance(geom, (bytes, bytearray, memoryview)):\n        return wkb.loads(bytes(geom))\n    if isinstance(geom, dict):\n        return shape(geom)\n    return None\n\n\ndef roof_candidates(buildings, *, latitude: float, longitude: float, geotiff):\n    """Select mapped *building geometry*, not ODHF-site circles or grounds.\n\n    Prefer a building explicitly classified as a hospital. Fallback is at most\n    one physically nearby nonresidential building: very weak pseudo-label.\n    """\n    to_merc = Transformer.from_crs(\'EPSG:4326\', \'EPSG:3857\', always_xy=True)\n    reference = Point(*to_merc.transform(longitude, latitude))\n    physical_scale = max(.3, math.cos(math.radians(latitude)))\n    ranked = []\n    with rasterio.open(geotiff) as ds:\n        if ds.crs is None or ds.width < 100 or ds.height < 100:\n            raise ValueError(\'Source GeoTIFF missing georeference or invalid size\')\n        to_raster = Transformer.from_crs(\'EPSG:4326\', ds.crs, always_xy=True)\n        pixel = ~ds.transform\n        for item in buildings:\n            category = str(item.get(\'class\') or \'\').casefold().strip()\n            if category in EXCLUDE_CLASSES:\n                continue\n            geom = _geometry(item)\n            if geom is None or geom.is_empty:\n                continue\n            if isinstance(geom, MultiPolygon):\n                geom = max(geom.geoms, key=lambda p: p.area)\n            if not isinstance(geom, Polygon):\n                continue\n            if not geom.is_valid:\n                geom = geom.buffer(0)\n            if not isinstance(geom, Polygon) or geom.is_empty:\n                continue\n            projected = geom_transform(to_merc.transform, geom)\n            area_m2 = projected.area * physical_scale ** 2\n            if not 45 <= area_m2 <= 60000:\n                continue\n            distance_m = projected.distance(reference) * physical_scale\n            explicit = category == \'hospital\'\n            if distance_m > (120 if explicit else 45):\n                continue\n            raster_poly = geom_transform(to_raster.transform, geom)\n            xys = [pixel * point for point in raster_poly.exterior.coords]\n            vals = [v for p in xys for v in p]\n            if not all(math.isfinite(v) for v in vals):\n                continue\n            x1 = max(0, min(p[0] for p in xys))\n            x2 = min(ds.width, max(p[0] for p in xys))\n            y1 = max(0, min(p[1] for p in xys))\n            y2 = min(ds.height, max(p[1] for p in xys))\n            if x2-x1 < 7 or y2-y1 < 7 or (x2-x1)*(y2-y1) > ds.width*ds.height*.75:\n                continue\n            evidence = 2 if explicit else 1\n            ranked.append((0 if explicit else 1, distance_m, {\n                \'osm_id\': \'overture/\' + str(item.get(\'id\') or \'\'),\n                \'bbox\': [round(x1, 1),round(y1,1),round(x2,1),round(y2,1)],\n                \'evidence_level\': evidence,\n                \'reason\': \'overture_building_class_hospital\' if explicit else \'overture_nearest_footprint_to_odhf_reference_only\',\n                \'point_distance_m\': round(distance_m,1),\n                \'source\': \'Overture Maps buildings\',\n                \'warning\': \'UNVERIFIED use; nearest building is not proof of hospital identity\',\n                \'overture_class\': category,\n            }))\n    ranked.sort(key=lambda row:(row[0],row[1]))\n    hospitals = [obj for level,_,obj in ranked if level == 0]\n    return hospitals[:3] if hospitals else [ranked[0][2]] if ranked else []\n\n\ndef generate_overture_drafts(root, limit=50, target_sites=8, provider: Callable=fetch_overture_bbox):\n    """Bounded fallback with per-image checkpoints, no simulated/approved labels.\n\n    Stops after enough independent draft sites have been collected. Fail-fast\n    on remote access error, so Colab doesn\'t spin through 50 dead requests.\n    """\n    root=Path(root).resolve()\n    if not 1<=limit<=100 or not 1<=target_sites<=30:\n        raise ValueError(\'Invalid candidate limits\')\n    manifest = root/\'review_manifest.csv\'\n    if not manifest.is_file():\n        raise FileNotFoundError(\'Saved review manifest is missing\')\n    df = pd.read_csv(manifest, keep_default_na=False)\n    wanted = {\'tile_id\',\'site_id\',\'image\',\'geotiff\',\'latitude\',\'longitude\',\'review_status\',\'candidate_class\'}\n    if not wanted.issubset(df.columns):\n        raise ValueError(\'Review manifest missing essential columns\')\n    df=df.loc[df.candidate_class.eq(\'known_hospital_site\') & df.review_status.eq(\'UNREVIEWED\')].head(limit)\n    dest=root/\'drafts\'/\'overture\';dest.mkdir(parents=True,exist_ok=True)\n    success_sites=set(); used=0; cached=0; errors=[]; proposals=0\n    for row in df.itertuples(index=False):\n        tile=str(row.tile_id); site=str(row.site_id)\n        if not tile or \'/\' in tile or \'..\' in tile:\n            continue\n        image=(root/str(row.image)).resolve(); tif=(root/str(row.geotiff)).resolve()\n        if not image.is_relative_to(root) or not tif.is_relative_to(root) or not image.is_file() or not tif.is_file():\n            continue\n        # Preserve any positive OpenStreetMap proposals; never modify them.\n        osm=root/\'drafts\'/\'osm\'/f\'{tile}.json\'\n        if osm.is_file():\n            try:\n                prev=json.loads(osm.read_text())\n                if prev.get(\'approved\') is False and any(int(c.get(\'evidence_level\',0))>=1 for c in prev.get(\'candidates\',[])):\n                    success_sites.add(site)\n                    continue\n            except (ValueError,TypeError,OSError):\n                pass\n        out=dest/(tile+\'.json\')\n        if out.is_file():\n            try:\n                doc=json.loads(out.read_text())\n                if doc.get(\'tile_id\')==tile and doc.get(\'approved\') is False and doc.get(\'candidates\'):\n                    success_sites.add(site); cached += 1\n                    if len(success_sites)>=target_sites:\n                        break\n                    continue\n            except (ValueError,TypeError,OSError):pass\n        if len(success_sites)>=target_sites:\n            break\n        try:\n            with rasterio.open(tif) as ds:\n                extent=transform_bounds(ds.crs,\'EPSG:4326\',*ds.bounds, densify_pts=10)\n            print(f\'Overture fallback {used+1}: checking saved site {tile}...\',flush=True)\n            elements=provider(extent)\n            used += 1\n            candidates=roof_candidates(elements,latitude=float(row.latitude),longitude=float(row.longitude),geotiff=tif)\n            data={\'tile_id\':tile,\'site_id\':site,\'approved\':False,\'verification\':\'NOT_GROUND_TRUTH\',\n                  \'source\':\'Overture Maps buildings + Statistics Canada ODHF reference proximity\',\n                  \'license_url\':LICENSE_URL,\'attribution\':ATTRIBUTION,\n                  \'created_at_utc\':datetime.now(timezone.utc).isoformat(),\n                  \'warning\':\'UNVERIFIED Overture roof hypothesis; NOT a confirmed hospital building\',\n                  \'candidates\':candidates}\n            tmp=out.with_suffix(\'.json.tmp\');tmp.write_text(json.dumps(data,indent=2),encoding=\'utf-8\');tmp.replace(out)\n            if candidates:\n                success_sites.add(site)\n                proposals += len(candidates)\n            print(f\'Overture complete: {len(success_sites)} independent candidate sites, {len(candidates)} new roof boxes.\',flush=True)\n        except Exception as exc:\n            msg=f\'{type(exc).__name__}: {str(exc)[:210]}\'\n            errors.append(f\'{tile}: {msg}\')\n            print(\'Overture unavailable; stopping rather than retrying every image: \'+msg,flush=True)\n            break\n    return {\'source\':\'Overture Maps\',\'candidate_sites_with_proposals\':len(success_sites),\n            \'requests\':used,\'cached\':cached,\'new_proposals\':proposals,\'errors\':errors,\n            \'status\':\'DRAFTS_ONLY_NOT_VERIFIED\'}\n'
for name, source in [('hospital_ai', HOSPITAL_AI_SOURCE),('osm_auto_annotations', OSM_AUTO_SOURCE),('hospital_review', HOSPITAL_REVIEW_SOURCE),('overture_weak_labels', OVERTURE_WEAK_SOURCE)]:
    path=Path('/content')/(name+'.py')
    path.write_text(source,encoding='utf-8')
    spec=importlib.util.spec_from_file_location(name,str(path))
    module=importlib.util.module_from_spec(spec)
    sys.modules[name]=module
    spec.loader.exec_module(module)
import hospital_ai as ai
import hospital_review as reviewer
import osm_auto_annotations as osm_auto
import overture_weak_labels as overture_auto
source_dir=ROOT/'source';source_dir.mkdir(parents=True,exist_ok=True)
for name,src in [('hospital_ai',HOSPITAL_AI_SOURCE),('hospital_review',HOSPITAL_REVIEW_SOURCE),('osm_auto_annotations',OSM_AUTO_SOURCE),('overture_weak_labels',OVERTURE_WEAK_SOURCE)]:
    (source_dir/(name+'.py')).write_text(src,encoding='utf-8')
print('Training, OpenStreetMap and Overture weak-label readers loaded. No model trained in setup.')


## 3 — Get the official facility reference CSV
Uses the same Statistics Canada ODHF source as the map. If you have already run Notebook 04 in this runtime, its CSV is reused. If not, it is downloaded here. References are candidate site locators only.

In [ ]:
PREPARATION_SOURCE = '"""Prepare genuine Canadian healthcare reference locations for the Colab map.\n\nPreferred: Statistics Canada ODHF v2 (2026) official GeoJSON archive.\nBackup: Government of Canada ODHF v1 (2020) ArcGIS REST service when v2\ncannot be downloaded (e.g., HTTP 403). Backup has older type classifications\nand no trustworthy emergency-services indicator. The source is always labelled.\nNo data is fabricated. Reference facilities are not machine-learning detections.\n"""\nfrom __future__ import annotations\nfrom io import BytesIO\nfrom pathlib import Path\nimport json\nimport zipfile\nimport pandas as pd\nimport requests\nfrom pyproj import Transformer\nfrom shapely.geometry import shape\n\nDATA_FILE = Path(\'/content/odhf_facilities.csv\') if Path(\'/content\').exists() else Path(\'odhf_facilities.csv\')\nMETA_FILE = DATA_FILE.with_suffix(\'.source.json\')\nV2_URL = \'https://www150.statcan.gc.ca/n1/pub/13-26-0001/2020001/zip/odhf_v2_geojson.zip\'\n# Official Canadian government public map-service layer for ODHF v1, exposed as JSON.\nV1_REST_URLS = (\n    \'https://www.earthquakescanada.nrcan.gc.ca/server2_serveur2/rest/services/StatCan/OpenDatabaseHealthFacilities/MapServer/0/query\',\n    \'https://maps-cartes.services.geo.ca/server2_serveur2/rest/services/StatCan/OpenDatabaseHealthFacilities/MapServer/0/query\',\n)\nHEADERS = {\'User-Agent\':\'Mozilla/5.0 (compatible; OSS4008-Research-Project/1.0)\',\n           \'Accept\':\'application/json,application/zip,*/*\'}\nCOLUMNS = [\'id\',\'name\',\'latitude\',\'longitude\',\'type\',\'address\',\'province\',\'emergency_services\',\'ownership\',\'source_version\']\n\n\ndef _normalize(rows, version):\n    if not rows:\n        raise ValueError(\'The source returned zero valid healthcare facilities.\')\n    df = pd.DataFrame.from_records(rows)\n    for col in COLUMNS:\n        if col not in df:\n            df[col] = \'\'\n    df[\'latitude\'] = pd.to_numeric(df[\'latitude\'], errors=\'coerce\')\n    df[\'longitude\'] = pd.to_numeric(df[\'longitude\'], errors=\'coerce\')\n    df = df.loc[df.latitude.between(41,84) & df.longitude.between(-142,-52)].copy()\n    df = df.drop_duplicates(subset=[\'name\',\'latitude\',\'longitude\',\'type\'])\n    df[\'source_version\'] = version\n    if df.empty:\n        raise ValueError(\'No source coordinates passed the Canadian geographic validation.\')\n    return df[COLUMNS].reset_index(drop=True)\n\n\ndef _from_v2(session):\n    response = session.get(V2_URL, timeout=45, headers=HEADERS)\n    response.raise_for_status()\n    if len(response.content)>300_000_000 or not zipfile.is_zipfile(BytesIO(response.content)):\n        raise ValueError(\'Invalid or oversized ODHF v2 archive\')\n    transformer = Transformer.from_crs(\'EPSG:3347\',\'EPSG:4326\',always_xy=True)\n    records=[]\n    with zipfile.ZipFile(BytesIO(response.content)) as archive:\n        files=[name for name in archive.namelist()\n               if name.lower().endswith((\'.geojson\',\'.json\')) and not name.startswith(\'__MACOSX/\')]\n        if not files:\n            raise ValueError(\'The official v2 ZIP does not contain a GeoJSON file\')\n        for filename in files:\n            content=json.loads(archive.read(filename))\n            for feature in content.get(\'features\',[]):\n                p=feature.get(\'properties\') or {}\n                geom=feature.get(\'geometry\')\n                if not geom:\n                    continue\n                try:\n                    g=shape(geom)\n                    if g.is_empty:\n                        continue\n                    pt=g.representative_point()\n                    lon,lat=float(pt.x),float(pt.y)\n                    if abs(lon)>180 or abs(lat)>90:\n                        lon,lat=transformer.transform(lon,lat)\n                    if not (41<=lat<=84 and -142<=lon<=-52):\n                        continue\n                except (TypeError,ValueError):\n                    continue\n                records.append({\n                    \'id\':str(p.get(\'id\') or \'\'),\'name\':str(p.get(\'name\') or \'Unnamed facility\'),\n                    \'latitude\':lat,\'longitude\':lon,\'type\':str(p.get(\'type\') or \'Unspecified\'),\n                    \'address\':str(p.get(\'address\') or \'\'),\n                    \'province\':str(p.get(\'prov_terr\') or \'\'),\n                    \'emergency_services\':str(p.get(\'emergency_services\') or \'\'),\n                    \'ownership\':str(p.get(\'ownership\') or \'\'),\n                })\n    return _normalize(records, \'ODHF v2 (2026)\')\n\n\ndef _from_v1_service(session, service_url):\n    # Object-ID pagination avoids ArcGIS maxRecordCount truncation.\n    params={\'where\':\'1=1\',\'returnIdsOnly\':\'true\',\'f\':\'json\'}\n    response=session.get(service_url,params=params,timeout=35,headers=HEADERS)\n    response.raise_for_status()\n    index=response.json()\n    if index.get(\'error\'):\n        raise RuntimeError(f"ArcGIS error: {index[\'error\']}")\n    ids=index.get(\'objectIds\') or []\n    if not ids:\n        raise ValueError(\'ODHF v1 service returned no Object IDs\')\n    if len(ids)>30000:\n        raise ValueError(\'Unexpectedly large ODHF v1 response\')\n    rows=[]\n    for start in range(0,len(ids),300):\n        chunk=ids[start:start+300]\n        r=session.get(service_url,params={\n            \'objectIds\':\',\'.join(map(str,chunk)),\n            \'outFields\':\'OBJECTID,name_nom,facility_type,latitude,longitude,address,province_en\',\n            \'returnGeometry\':\'false\', \'f\':\'json\'},timeout=35,headers=HEADERS)\n        r.raise_for_status()\n        result=r.json()\n        if result.get(\'error\'):\n            raise RuntimeError(f"ArcGIS query error: {result[\'error\']}")\n        for feat in result.get(\'features\',[]):\n            p=feat.get(\'attributes\') or {}\n            rows.append({\n                \'id\':str(p.get(\'OBJECTID\') or \'\'),\n                \'name\':str(p.get(\'name_nom\') or \'Unnamed facility\'),\n                \'latitude\':p.get(\'latitude\'),\'longitude\':p.get(\'longitude\'),\n                \'type\':str(p.get(\'facility_type\') or \'Unspecified\'),\n                \'address\':str(p.get(\'address\') or \'\'),\n                \'province\':str(p.get(\'province_en\') or \'\'),\n                \'emergency_services\':\'\',\'ownership\':\'\'})\n    # Do not accept partially paged data as if it were the national source.\n    if len(rows) < max(1,int(len(ids)*0.95)):\n        raise RuntimeError(f\'Incomplete ODHF v1 retrieval: {len(rows)} of {len(ids)} IDs\')\n    return _normalize(rows, \'ODHF v1 (2020 fallback)\')\n\n\ndef build_dataset(force=False, allow_v1_fallback=True, session=None):\n    """Download official reference records, use verified cache, or explain failure.\n\n    Returns (DataFrame, metadata dict). Never invents records.\n    """\n    if not force and DATA_FILE.is_file():\n        df=pd.read_csv(DATA_FILE, low_memory=False).fillna(\'\')\n        if {\'latitude\',\'longitude\',\'name\',\'type\'}.issubset(df.columns) and len(df):\n            meta=json.loads(META_FILE.read_text()) if META_FILE.exists() else {\n                \'source_version\':str(df.get(\'source_version\',pd.Series([\'Existing reference cache\'])).iloc[0]),\n                \'source_url\':\'Existing local CSV\', \'note\':\'Cached reference records; source age may be unknown.\'}\n            print(f\'Using {len(df):,} cached reference facilities: {meta["source_version"]}\')\n            return df,meta\n    client=session or requests.Session()\n    errors=[]\n    try:\n        df=_from_v2(client)\n        meta={\'source_version\':\'ODHF v2 (2026)\',\'source_url\':V2_URL,\n              \'note\':\'Official Statistics Canada 2026 reference records; not AI predictions.\'}\n    except (requests.RequestException,ValueError,RuntimeError,zipfile.BadZipFile,KeyError) as e:\n        errors.append(\'2026 source: \'+str(e)[:200])\n        if not allow_v1_fallback:\n            raise RuntimeError(\'ODHF v2 unavailable; no fake records were created. \'+\'; \'.join(errors)) from e\n        df=None\n        meta=None\n        for endpoint in V1_REST_URLS:\n            try:\n                df=_from_v1_service(client,endpoint)\n                meta={\'source_version\':\'ODHF v1 (2020 fallback)\',\'source_url\':endpoint,\n                      \'note\':\'Older official Canadian reference data. No reliable emergency-services or fee fields.\'}\n                break\n            except (requests.RequestException,ValueError,RuntimeError,KeyError) as backup_err:\n                errors.append(\'backup: \'+str(backup_err)[:200])\n        if df is None:\n            raise RuntimeError(\'No official hospital reference source could be downloaded. \'\n                               \'Internet/network restrictions may be blocking both sources. \'\n                               +\'; \'.join(errors))\n    DATA_FILE.parent.mkdir(parents=True,exist_ok=True)\n    df.to_csv(DATA_FILE,index=False)\n    META_FILE.write_text(json.dumps(meta,indent=2),encoding=\'utf-8\')\n    print(f\'Saved {len(df):,} official reference facilities to {DATA_FILE}.\')\n    print(\'Source:\',meta[\'source_version\'])\n    if errors:\n        print(\'Note: ODHF v2 download failed; using the older official reference layer.\')\n    return df,meta\n\nif __name__==\'__main__\':\n    build_dataset()\n'
Path('/content/prepare_odhf.py').write_text(PREPARATION_SOURCE,encoding='utf-8')
spec=importlib.util.spec_from_file_location('oss_odhf','/content/prepare_odhf.py')
refmod=importlib.util.module_from_spec(spec)
sys.modules['oss_odhf']=refmod
spec.loader.exec_module(refmod)
try:
    facility_data, source_meta = refmod.build_dataset()
    print('Reference facilities:', len(facility_data), 'source:',source_meta.get('source_version'))
except Exception as e:
    facility_data=None
    print('BLOCKED: Could not load source data:',type(e).__name__,str(e)[:300])

# Maintain persistent official reference cache for the UI-only map launcher.
if facility_data is not None and Path('/content/odhf_facilities.csv').is_file():
    import shutil
    cache=ROOT/'reference_cache';cache.mkdir(parents=True,exist_ok=True)
    shutil.copy2('/content/odhf_facilities.csv',cache/'odhf_facilities.csv')
    meta_path=Path('/content/odhf_facilities.source.json')
    if meta_path.exists():shutil.copy2(meta_path,cache/'odhf_facilities.source.json')
    print('Reference cache ready for Notebook 04')


## 4 — Download reviewable Ontario aerial chips
This selects a small number of *known hospital sites* and *other healthcare facilities for comparison*. **No tile is automatically labeled a hospital or nonhospital rooftop.** Review the imagery and verify building use before marking approved.

Default Ottawa pilot: 12 hospital-reference locations and 12 comparison candidates, 320-meter ground-square, 640×640 pixels (roughly 0.5 m/px). Imagery coverage varies; failures go into `imagery_errors.json`. If the official Ontario image service rejects requests, training remains blocked rather than using unlicensed base-map screenshots.

In [ ]:
HOSPITAL_CHIPS = 12
COMPARISON_CHIPS = 12
if facility_data is not None:
    try:
        manifest_file=ROOT/'review_manifest.csv'
        if manifest_file.exists():
            import pandas as pd
            preview_manifest=pd.read_csv(manifest_file,keep_default_na=False)
            print('Reusing existing annotation manifest:',len(preview_manifest),'chips. Not overwriting human review.')
        else:
            preview_manifest=ai.prepare_review_chips('/content/odhf_facilities.csv',ROOT,
                center_lat=45.4215,center_lon=-75.6972,radius_km=30,
                hospitals=HOSPITAL_CHIPS,controls=COMPARISON_CHIPS)
            print('Downloaded',len(preview_manifest),'candidate images for HUMAN REVIEW')
        from IPython.display import display
        display(preview_manifest[['name','candidate_class','review_status']].head(25))
        from IPython.display import Image as NotebookImage
        display(NotebookImage(filename=str(ai.make_candidate_contact_sheet(ROOT))))
    except Exception as e:
        print('IMAGE PREPARATION BLOCKED:',type(e).__name__,str(e)[:500])
else:
    print('Skipping image acquisition: reference CSV unavailable.')

import json
print('IMAGERY DIAGNOSTIC:',json.dumps(ai.review_workspace_status(ROOT),indent=2))


## 4b — Optional OSM drafts from ALREADY SAVED images

This stage **does not re-download imagery**. Existing human approvals are never replaced. Set `RUN_OSM_DRAFTS=True` to perform/resume draft matching; results are cached in Google Drive. Public OSM services may be slow or rate-limited. **Unreviewed suggestions are not training ground truth.**

**Speed settings:** At most **50 existing images**, processed in groups of **4** with a visible message before and after each request. Previously downloaded images and cached OSM drafts are reused. Public requests have shorter timeouts and a second provider on connection failure; if the server is unavailable, the workflow stops and preserves completed drafts.


In [ ]:
# @title Optional: Resume OSM draft boxes from saved GeoTIFFs (no imagery downloads)
RUN_OSM_DRAFTS = False  # Change to True only when you want to run the remaining Overpass draft queries
OSM_TARGET_IMAGES = 50
if RUN_OSM_DRAFTS:
    import json
    from IPython.display import display,Image as DisplayImage
    report=osm_auto.generate_osm_drafts(ROOT,limit=OSM_TARGET_IMAGES,batch_size=4)
    print('OSM AUTOMATIC DRAFT SUMMARY:',json.dumps(report,indent=2))
    if report.get('images_with_drafts'):
        display(DisplayImage(filename=str(osm_auto.make_draft_contact_sheet(ROOT,max_items=36))))
else:
    print('OSM batch skipped (saved images and previous drafts unchanged).')
    previous=ROOT/'drafts'/'osm_inventory.json'
    if previous.exists():
        import json
        print('Previous cached OSM result:',json.dumps(json.loads(previous.read_text()),indent=2)[:1800])


## Hands-free OSM-matched experiment — NO manual drawing

This stage uses **existing, saved aerial GeoTIFFs** and nearby OpenStreetMap **building footprints** to generate pseudo-labels. The first pass checks **at most 50 saved images** in small, progress-reported batches using a secondary OSM provider after timeouts (not 125). It **never approves** labels in the reviewed training dataset. It only includes candidate hospital roofs with OSM building-tag evidence and skips unsupported or ambiguous sites. You do **not** need to draw any rectangles.

**It is an experimental, weakly supervised model.** Its evaluation uses OSM-derived proxy labels, so reported precision/mAP is **not independently validated hospital-identification accuracy**. Training automatically stops if too few sites qualify.

- Set `RUN_HANDS_FREE_AI = True` below and run **only that cell**. This resumes draft lookups if needed, prepares a separate YOLO dataset, trains (if sufficient sites), then exports **unverified candidate markers** to the map's saved Drive folder.
- When it has finished, open [Notebook 04 — Final Map](https://colab.research.google.com/drive/1YYpGBo-imloznVD9yRzQ4zDmC7EwVOuJ) and choose Run all. Notebook 04 itself never downloads images or retrains.
- If a trained weak checkpoint already matches the dataset, it is reused; no duplicate training unless you explicitly ask to retrain.

**Fallback when Overpass is unavailable:** The official Overture Maps building dataset is queried for the same georeferenced image chips, stopping once eight independent experimental candidates are collected (up to 50 tested). This is a proximity-based *weak label*, not proof of hospital function or measured detection accuracy. If Overture cloud access also fails, training remains blocked.


In [ ]:
# @title Optional ONE-CELL hands-free OSM matching + experimental YOLO training
RUN_HANDS_FREE_AI = True  # Set to True for a deliberate run; leave False for normal Run all
RUN_PUBLIC_OVERPASS = False  # Both public Overpass endpoints timed out in live Colab; prefer Overture fallback
OSM_WEAK_MIN_EVIDENCE = 1   # Include building footprints mapped inside hospital grounds; weak and unverified
OSM_WEAK_TARGET_IMAGES = 50  # Upper cap; existing downloads and cached drafts reused
OSM_WEAK_BATCH_SIZE = 2   # Smaller, faster OSM queries; two public providers if primary times out
RETRAIN_WEAK_MODEL = False  # Default: reuse saved trained model
WEAK_TRAIN_EPOCHS = 12

WEAK_OSM_SOURCE = '"""Hands-free *experimental* OSM weak supervision for hospital-roof candidates.\n\nNO human-verified labels are created or changed. A weak model can learn from\ngeographically aligned building polygons with OSM hospital evidence, but OSM\nlabels are NOT independent ground truth. Proxy metrics are NEVER real-world\nhospital detection accuracy. Results never overwrite the reviewed detector.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport random\nimport shutil\nfrom collections import Counter\nfrom pathlib import Path\n\nimport pandas as pd\nimport yaml\nfrom PIL import Image\n\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\n\n\ndef box_to_yolo(box,w,h):\n    if not isinstance(box,(list,tuple)) or len(box)!=4 or w<=0 or h<=0:raise ValueError(\'Invalid box or image size\')\n    x1,y1,x2,y2=[float(n) for n in box]\n    if not all(math.isfinite(v) for v in [x1,y1,x2,y2]):raise ValueError(\'Nonfinite box\')\n    x1=max(0,min(w,x1)); x2=max(0,min(w,x2))\n    y1=max(0,min(h,y1)); y2=max(0,min(h,y2))\n    if x2-x1<6 or y2-y1<6:raise ValueError(\'Too-small box\')\n    return f\'0 {(x1+x2)/(2*w):.7f} {(y1+y2)/(2*h):.7f} {(x2-x1)/w:.7f} {(y2-y1)/h:.7f}\'\n\n\ndef make_weak_dataset(root, *, min_evidence=2, minimum_sites=8, seed=4008):\n    """Convert only OSM draft roof boxes to isolated pseudo-label YOLO dataset.\n\n    min_evidence 3: explicit amenity/healthcare=hospital on building.\n    min_evidence 2: additionally building=hospital (could be repurposed).\n    min_evidence 1: additionally buildings inside hospital campus (higher risk).\n    No generic distance-only evidence 0 is included. Site-wise train/val/test.\n    """\n    root=Path(root).resolve(); manifest=root/\'review_manifest.csv\'\n    if min_evidence not in (1,2,3):raise ValueError(\'min_evidence must be 1,2,3\')\n    if not 6<=minimum_sites<=100:raise ValueError(\'minimum_sites must be 6..100\')\n    if not manifest.is_file():raise FileNotFoundError(\'Missing review_manifest.csv\')\n    df=pd.read_csv(manifest,keep_default_na=False)\n    required={\'tile_id\',\'site_id\',\'image\',\'geotiff\',\'candidate_class\',\'review_status\',\'imagery_license_url\'}\n    if not required.issubset(df):raise ValueError(\'Manifest missing fields: \'+str(sorted(required-set(df))))\n    if df.tile_id.duplicated().any():raise ValueError(\'Duplicate tile IDs\')\n    records=[];counts=Counter()\n    for _,r in df.iterrows():\n        # preserve and exclude human-reviewed data; no autogenerated ground truth.\n        if r.review_status!=\'UNREVIEWED\' or r.candidate_class!=\'known_hospital_site\':continue\n        tile=str(r.tile_id)\n        if \'/\' in tile or \'..\' in tile:continue\n        ann=root/\'drafts\'/\'osm\'/(tile+\'.json\')\n        ov=root/\'drafts\'/\'overture\'/(tile+\'.json\')\n        # Prefer a usable OSM candidate. A cached empty/near-reference-only OSM\n        # draft must not prevent legitimate Overture data from being considered.\n        try:\n            osm_doc=json.loads(ann.read_text(encoding=\'utf-8\')) if ann.is_file() else {}\n            usable_osm=any(int(c.get(\'evidence_level\',-1))>=min_evidence for c in osm_doc.get(\'candidates\',[]))\n        except (ValueError,OSError,TypeError):\n            usable_osm=False\n        if (not usable_osm) and ov.is_file():\n            ann=ov\n        if not ann.is_file():counts[\'no_map_draft\']+=1;continue\n        try:\n            doc=json.loads(ann.read_text(encoding=\'utf-8\'))\n            if doc.get(\'tile_id\')!=tile or doc.get(\'approved\') is not False:raise ValueError(\'Unacceptable draft status\')\n            image=(root/str(r.image)).resolve(); tif=(root/str(r.geotiff)).resolve()\n            if not image.is_relative_to(root) or not tif.is_relative_to(root) or not image.is_file() or not tif.is_file():\n                raise ValueError(\'Image/GeoTIFF missing or outside root\')\n            with Image.open(image) as picture:width,height=picture.size\n            lines=[];evidence=[];ids=set()\n            for c in doc.get(\'candidates\',[]):\n                lvl=int(c.get(\'evidence_level\',-1))\n                if not min_evidence<=lvl<=3:continue\n                if lvl==3 and c.get(\'reason\')!=\'explicit_hospital_use_tag\':continue\n                if lvl==2 and c.get(\'reason\') not in (\'building_hospital_tag_not_confirmed_current_use\',\'overture_building_class_hospital\'):continue\n                if lvl==1 and c.get(\'reason\') not in (\'building_inside_mapped_hospital_campus\',\'overture_nearest_footprint_to_odhf_reference_only\'):continue\n                osm_id=str(c.get(\'osm_id\',\'\'))\n                if not (osm_id.startswith(\'way/\') or osm_id.startswith(\'relation/\') or osm_id.startswith(\'overture/\')) or osm_id in ids:continue\n                ids.add(osm_id)\n                lines.append(box_to_yolo(c[\'bbox\'],width,height));evidence.append({\'osm_id\':osm_id,\'evidence_level\':lvl,\'reason\':c.get(\'reason\',\'\')})\n            if not lines:counts[\'no_strong_roof_evidence\']+=1;continue\n            if not str(r.imagery_license_url).startswith(\'https://\'):raise ValueError(\'Missing recorded imagery source license URL\')\n            records.append({\'tile_id\':tile,\'site_id\':str(r.site_id),\'image\':image,\'lines\':lines,\n                            \'evidence\':evidence,\'draft_source\':str(doc.get(\'source\',\'unknown\')),\'imagery_source\':str(r.get(\'imagery_source\',\'\')),\n                            \'imagery_license_url\':str(r.imagery_license_url),\'draft_path\':ann})\n            counts[\'matched_images\']+=1\n        except (ValueError,TypeError,KeyError,OSError,json.JSONDecodeError) as exc:\n            counts[\'invalid_record\']+=1\n            continue\n    groups={}\n    for r in records:groups.setdefault(r[\'site_id\'],[]).append(r)\n    if len(groups)<minimum_sites:\n        summary={\'status\':\'INSUFFICIENT_PSEUDO_LABEL_SITES\',\'candidate_images\':int(len(df)),\'matched_images\':len(records),\n                 \'independent_sites\':len(groups),\'minimum_sites\':minimum_sites,\'minimum_evidence\':min_evidence,\n                 \'counts\':dict(counts),\'training_started\':False,\n                 \'next\':\'Need more explicitly mapped hospital building footprints; do not automatically label OSM grounds or make up training labels.\'}\n        p=root/\'weak_osm_experiment\'/\'preparation_status.json\';p.parent.mkdir(parents=True,exist_ok=True)\n        p.write_text(json.dumps(summary,indent=2))\n        return summary\n    rng=random.Random(seed);sids=sorted(groups);rng.shuffle(sids)\n    n=len(sids);n_test=max(2,int(n*.2));n_val=max(2,int(n*.2));\n    if n-n_test-n_val<2:raise ValueError(\'Not enough training groups\')\n    by_site={sid:(\'test\' if i<n_test else \'val\' if i<n_test+n_val else \'train\') for i,sid in enumerate(sids)}\n    dest=root/\'weak_osm_experiment\'/\'dataset\';dest.mkdir(parents=True,exist_ok=True)\n    # refresh only this isolated weak dataset; never touch verified label files.\n    for fold in [\'images\',\'labels\']:\n        shutil.rmtree(dest/fold,ignore_errors=True)\n    rows=[]\n    for r in records:\n        split=by_site[r[\'site_id\']]\n        img=dest/\'images\'/split/r[\'image\'].name\n        lbl=dest/\'labels\'/split/(r[\'image\'].stem+\'.txt\')\n        img.parent.mkdir(parents=True,exist_ok=True);lbl.parent.mkdir(parents=True,exist_ok=True)\n        shutil.copy2(r[\'image\'],img);lbl.write_text(\'\\n\'.join(r[\'lines\'])+\'\\n\',encoding=\'utf-8\')\n        rows.append({\'site_id\':r[\'site_id\'],\'tile_id\':r[\'tile_id\'],\'split\':split,\'num_pseudo_boxes\':len(r[\'lines\']),\n                     \'map_evidence\':json.dumps(r[\'evidence\'],separators=(\',\',\':\')),\'draft_source\':r[\'draft_source\'],\n                     \'image_sha256\':sha(img),\'osm_draft_sha256\':sha(r[\'draft_path\']),\n                     \'imagery_source\':r[\'imagery_source\'],\'imagery_license_url\':r[\'imagery_license_url\']})\n    data={\'path\':str(dest),\'train\':\'images/train\',\'val\':\'images/val\',\'test\':\'images/test\',\'names\':{0:\'hospital_candidate\'}}\n    (dest/\'data.yaml\').write_text(yaml.safe_dump(data,sort_keys=False))\n    csv=dest/\'weak_site_split_manifest.csv\';pd.DataFrame(rows).to_csv(csv,index=False)\n    summary={\'status\':\'WEAK_PSEUDO_DATASET_READY_NOT_GROUND_TRUTH\',\'independent_sites\':n,\n        \'matched_images\':len(records),\'pseudo_boxes\':sum(len(r[\'lines\']) for r in records),\n        \'train_sites\':sum(v==\'train\' for v in by_site.values()),\n        \'val_sites\':sum(v==\'val\' for v in by_site.values()),\'test_sites\':sum(v==\'test\' for v in by_site.values()),\n        \'minimum_evidence\':min_evidence,\'manifest_sha256\':sha(csv),\'data_yaml\':str(dest/\'data.yaml\'),\n        \'counts\':dict(counts),\'training_started\':False,\n        \'disclaimer\':\'Training labels come from OSM or Overture building polygons and ODHF reference proximity, NEVER independent hospital identity verification. Metrics are map-derived proxy only, not real accuracy.\'}\n    (dest.parent/\'preparation_status.json\').write_text(json.dumps(summary,indent=2))\n    return summary\n\n\ndef train_weak_experiment(root,*,epochs=12,batch=8,imgsz=640,pretrained=\'yolo11n.pt\'):\n    """GPU train against OSM pseudo labels. Reports PROXY metrics only."""\n    root=Path(root);experiment=root/\'weak_osm_experiment\';info=json.loads((experiment/\'preparation_status.json\').read_text())\n    if info.get(\'status\')!=\'WEAK_PSEUDO_DATASET_READY_NOT_GROUND_TRUTH\':\n        raise ValueError(\'Not enough OSM pseudo labels; no experimental training permitted\')\n    if not 1<=epochs<=100 or not 1<=batch<=64:raise ValueError(\'Invalid training options\')\n    if not (experiment/\'dataset\'/\'weak_site_split_manifest.csv\').exists():raise ValueError(\'Missing split\')\n    if sha(experiment/\'dataset\'/\'weak_site_split_manifest.csv\')!=info[\'manifest_sha256\']:\n        raise ValueError(\'Weak dataset changed since preparation\')\n    from ultralytics import YOLO\n    import torch\n    model=YOLO(pretrained)\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'GPU not available: in Notebook 08 select Runtime > Change runtime type > T4 GPU before starting training. No slow CPU training attempted.\')\n    device=0\n    fit=model.train(data=info[\'data_yaml\'],epochs=epochs,imgsz=imgsz,batch=batch,\n        device=device,seed=4008,project=str(experiment/\'runs\'),name=\'weak_hospital\',exist_ok=True,patience=6)\n    weights=Path(fit.save_dir)/\'weights\'/\'best.pt\'\n    if not weights.is_file():raise RuntimeError(\'No weights after YOLO training\')\n    out=experiment/\'weak_best.pt\';shutil.copy2(weights,out)\n    ev=YOLO(str(out)).val(data=info[\'data_yaml\'],split=\'test\',imgsz=imgsz,device=device,\n                             project=str(experiment/\'runs\'),name=\'weak_test\',exist_ok=True)\n    vals={\'precision_proxy\':float(ev.box.mp),\'recall_proxy\':float(ev.box.mr),\n          \'mAP50_proxy\':float(ev.box.map50),\'mAP50_95_proxy\':float(ev.box.map)}\n    if not all(math.isfinite(v) and 0<=v<=1 for v in vals.values()):raise ValueError(\'Invalid weak-label metrics\')\n    report={\'status\':\'TRAINED_ON_OSM_PSEUDO_LABELS_NOT_VERIFIED\', \'weights_sha256\':sha(out),\n        \'split_sha256\':info[\'manifest_sha256\'],\'training_image_count\':info[\'matched_images\'],\n        \'proxy_test_sites\':info[\'test_sites\'],\'model_weights\':str(out),\n        \'proxy_metrics\':vals,\'model_input\':pretrained,\'epochs\':epochs,\'device\':str(device),\n        \'disclaimer\':\'Measured only against independent-site OSM/Overture/ODHF weak pseudo-labels. No hospital identification accuracy claim.\'}\n    (experiment/\'weak_training_report.json\').write_text(json.dumps(report,indent=2))\n    return report\n\n\ndef export_weak_heldout_geojson(root, *, conf=0.35, max_images=30):\n    """Genuine model-inferred WGS84 points, clearly classified as WEAK/UNVERIFIED.\n\n    Never writes reviewed-detector artifacts or training_report.json. Its model\n    uses OSM pseudo labels and has not been evaluated on independent truth.\n    """\n    root=Path(root).resolve();exp=root/\'weak_osm_experiment\'\n    rep=json.loads((exp/\'weak_training_report.json\').read_text(encoding=\'utf-8\'))\n    weights=exp/\'weak_best.pt\';split_file=exp/\'dataset/weak_site_split_manifest.csv\'\n    if rep.get(\'status\')!=\'TRAINED_ON_OSM_PSEUDO_LABELS_NOT_VERIFIED\' or not weights.is_file() or not split_file.is_file():\n        raise ValueError(\'No trained OSM weak model\')\n    if sha(weights)!=rep.get(\'weights_sha256\') or sha(split_file)!=rep.get(\'split_sha256\'):\n        raise ValueError(\'Weak model or dataset changed: inference blocked\')\n    if not 0<conf<1 or not 1<=max_images<=100:raise ValueError(\'Invalid inference settings\')\n    from ultralytics import YOLO\n    import rasterio\n    from rasterio.transform import xy\n    from pyproj import Transformer\n    model=YOLO(str(weights))\n    man=pd.read_csv(root/\'review_manifest.csv\',keep_default_na=False)\n    splits=pd.read_csv(split_file,keep_default_na=False)\n    test=splits.loc[splits.split.eq(\'test\'),[\'tile_id\',\'site_id\']]\n    if test.empty:raise ValueError(\'No held-out pseudo-label images\')\n    chosen=test.merge(man[[\'tile_id\',\'site_id\',\'geotiff\']],on=[\'tile_id\',\'site_id\'],validate=\'one_to_one\')\n    features=[];processed=0\n    for _,row in chosen.head(max_images).iterrows():\n        path=(root/str(row.geotiff)).resolve()\n        if not path.is_relative_to(root) or not path.is_file():raise ValueError(\'Unsafe or missing GeoTIFF\')\n        with rasterio.open(path) as ds:\n            if not ds.crs or ds.count<3 or ds.dtypes[0]!=\'uint8\':raise ValueError(\'GeoTIFF unsuitable\')\n            arr=ds.read((1,2,3)).transpose(1,2,0)\n            from_raster=Transformer.from_crs(ds.crs,\'EPSG:4326\',always_xy=True)\n            prediction=model.predict(source=Image.fromarray(arr),conf=conf,verbose=False)\n            for result in prediction:\n                for b in result.boxes:\n                    if int(b.cls[0])!=0:continue\n                    x1,y1,x2,y2=[float(z) for z in b.xyxy[0].tolist()]\n                    if not all(math.isfinite(z) for z in (x1,x2,y1,y2)) or x2<=x1 or y2<=y1:continue\n                    score=float(b.conf[0])\n                    if not 0<=score<=1:continue\n                    lon,lat=from_raster.transform(*xy(ds.transform,(y1+y2)/2,(x1+x2)/2))\n                    if not -142<=lon<=-52 or not 41<=lat<=84:continue\n                    features.append({\'type\':\'Feature\',\'geometry\':{\'type\':\'Point\',\'coordinates\':[lon,lat]},\n                        \'properties\':{\'confidence\':score,\'method\':\'osm_weak_supervision_experimental\',\n                        \'weights_sha256\':rep[\'weights_sha256\'],\'test_site_id\':str(row.site_id),\n                        \'warning\':\'Unverified AI output from OSM pseudo labels, NOT verified hospital identity\'}})\n                    if len(features)>3000:raise ValueError(\'Too many map features\')\n            processed+=1\n    geo={\'type\':\'FeatureCollection\',\'features\':features,\'properties\':{\n        \'status\':\'UNVERIFIED_WEAKLY_SUPERVISED_MODEL\',\n        \'warning\':\'Open-map-derived pseudo labels; proxy evaluation cannot establish real hospital detection accuracy.\',\n        \'weights_sha256\':rep[\'weights_sha256\'],\'image_count\':processed}}\n    outfile=exp/\'experimental_unverified_predictions.geojson\'\n    temp=outfile.with_suffix(\'.tmp\')\n    temp.write_text(json.dumps(geo,indent=2),encoding=\'utf-8\');temp.replace(outfile)\n    return {\'status\':\'EXPERIMENTAL_UNVERIFIED_AI_OUTPUT\',\'images\':processed,\'predictions\':len(features),\n            \'geojson\':str(outfile),\'warning\':geo[\'properties\'][\'warning\']}\n\n\ndef infer_weak_in_selected_circle(root, latitude, longitude, radius_m, *, confidence=.35):\n    """Experimental REAL model inference only; hospital use never verified.\n\n    Drawn circle must be 20..250m inside Ontario with licensed image access.\n    This path cannot be mistaken for separately evaluated `hospital_ai`.\n    """\n    from tempfile import TemporaryDirectory\n    from importlib import util\n    from math import radians, cos, hypot, asin, sin, sqrt\n    from pyproj import Transformer\n    import rasterio\n    from rasterio.transform import xy\n    from PIL import Image\n    from ultralytics import YOLO\n    root=Path(root).resolve()\n    exp=root/\'weak_osm_experiment\'\n    report_file=exp/\'weak_training_report.json\'\n    weights=exp/\'weak_best.pt\'\n    split=exp/\'dataset/weak_site_split_manifest.csv\'\n    if not all(p.is_file() for p in (report_file,weights,split)):\n        raise RuntimeError(\'NO EXPERIMENTAL MODEL: training was not completed in Notebook 08\')\n    report=json.loads(report_file.read_text())\n    if (report.get(\'status\')!=\'TRAINED_ON_OSM_PSEUDO_LABELS_NOT_VERIFIED\'\n        or report.get(\'weights_sha256\')!=sha(weights)\n        or report.get(\'split_sha256\')!=sha(split)):\n        raise RuntimeError(\'Experimental checkpoint provenance mismatch\')\n    lat,lon,rad=float(latitude),float(longitude),float(radius_m)\n    if not (41.7<=lat<=57 and -95.2<=lon<=-73.8 and 20<=rad<=250 and 0<confidence<1):\n        raise ValueError(\'Experimental live AI requires an Ontario drawn circle with radius 20..250 metres\')\n    ai_path=root/\'source\'/\'hospital_ai.py\'\n    if not ai_path.is_file():\n        raise FileNotFoundError(\'Missing saved Ontario image downloader source. Run Notebook 08 setup once.\')\n    import sys\n    spec=util.spec_from_file_location(\'hospital_ai_for_weak_inference\',str(ai_path))\n    ai=util.module_from_spec(spec);sys.modules[spec.name]=ai;spec.loader.exec_module(ai)\n    direct=Transformer.from_crs(\'EPSG:4326\',\'EPSG:3857\',always_xy=True)\n    inverse=Transformer.from_crs(\'EPSG:3857\',\'EPSG:4326\',always_xy=True)\n    x,y=direct.transform(lon,lat)\n    positions=[(0,0)] if rad<=130 else [(u,v) for u in (-250,0,250) for v in (-250,0,250)]\n    model=YOLO(str(weights))\n    result=[]\n    with TemporaryDirectory(prefix=\'oss4008_weak_infer_\') as directory:\n        for idx,(dx,dy) in enumerate(positions):\n            tile_lon,tile_lat=inverse.transform(x+dx/max(cos(radians(lat)),.01),\n                                                y+dy/max(cos(radians(lat)),.01))\n            png=Path(directory)/f\'image{idx}.png\';tif=Path(directory)/f\'image{idx}.tif\'\n            ai.download_ontario_chip(tile_lat,tile_lon,png,tif)\n            with rasterio.open(tif) as ds:\n                image=Image.fromarray(ds.read((1,2,3)).transpose(1,2,0))\n                to_wgs84=Transformer.from_crs(ds.crs,\'EPSG:4326\',always_xy=True)\n                for detection in model.predict(source=image,conf=confidence,verbose=False):\n                    for box in detection.boxes:\n                        if int(box.cls[0])!=0:continue\n                        x1,y1,x2,y2=map(float,box.xyxy[0].tolist())\n                        if not (0<=x1<x2<=ds.width and 0<=y1<y2<=ds.height):continue\n                        px,py=xy(ds.transform,(y1+y2)/2,(x1+x2)/2)\n                        olon,olat=to_wgs84.transform(px,py)\n                        a=sin(radians(olat-lat)/2)**2+cos(radians(lat))*cos(radians(olat))*sin(radians(olon-lon)/2)**2\n                        if 6371008.8*2*asin(min(1,sqrt(a)))>rad:continue\n                        result.append({\'type\':\'Feature\',\'geometry\':{\'type\':\'Point\',\'coordinates\':[olon,olat]},\n                                       \'properties\':{\'confidence\':float(box.conf[0]),\n                                                     \'method\':\'osm_weak_supervision_experimental\',\n                                                     \'weights_sha256\':report[\'weights_sha256\'],\n                                                     \'warning\':\'EXPERIMENTAL roof candidate; NOT verified hospital identity\'}})\n    # Remove duplicate overlapping chip detections within 12m.\n    unique=[]\n    for feature in sorted(result,key=lambda f:-f[\'properties\'][\'confidence\']):\n        lon2,lat2=feature[\'geometry\'][\'coordinates\']\n        if all(hypot((lat2-u[\'geometry\'][\'coordinates\'][1])*111132,\n                     (lon2-u[\'geometry\'][\'coordinates\'][0])*111320*cos(radians(lat2)))>12 for u in unique):\n            unique.append(feature)\n    return {\'ok\':True,\'experimental\':True,\'features\':unique,\'count\':len(unique),\n            \'image_chips\':len(positions),\'model_sha256\':report[\'weights_sha256\'],\n            \'note\':\'This is trained image inference but training labels were OSM/Overture/ODHF weak proxies; NOT verified hospital accuracy.\'}\n'
if RUN_HANDS_FREE_AI:
    import importlib.util,sys,json
    from pathlib import Path
    source_dir=ROOT/'source';source_dir.mkdir(parents=True,exist_ok=True)
    weak_file=source_dir/'weak_osm_training.py';weak_file.write_text(WEAK_OSM_SOURCE,encoding='utf-8')
    spec=importlib.util.spec_from_file_location('weak_osm_training',str(weak_file))
    weak=importlib.util.module_from_spec(spec);sys.modules[spec.name]=weak;spec.loader.exec_module(weak)
    print('Step 1/3: reuse existing OSM/Overture draft polygons; no unnecessary re-downloads.')
    if RUN_PUBLIC_OVERPASS:
        try:
            draft=osm_auto.generate_osm_drafts(ROOT,limit=OSM_WEAK_TARGET_IMAGES,batch_size=OSM_WEAK_BATCH_SIZE)
            print('Real OSM draft coverage:',json.dumps({k:v for k,v in draft.items() if k!='errors'},indent=2))
            if draft.get('errors'): print('OSM service issues:',draft['errors'][:5])
        except Exception as err:
            print('OSM draft fetching failed:',type(err).__name__,str(err)[:180])
    else:
        print('Overpass disabled after repeated LIVE Colab failures. Reusing any saved drafts and trying Overture if needed.')
    print('Step 2/3: building isolated OSM pseudo-label dataset (no human approvals fabricated).')
    status=weak.make_weak_dataset(ROOT,min_evidence=OSM_WEAK_MIN_EVIDENCE,minimum_sites=8)
    if status['status']=='INSUFFICIENT_PSEUDO_LABEL_SITES':
        print('Overpass did not supply enough eligible roof shapes. Trying INDEPENDENT Overture Maps building dataset, not another Overpass server.',flush=True)
        try:
            import overture_weak_labels as overture
            alt=overture.generate_overture_drafts(ROOT,limit=OSM_WEAK_TARGET_IMAGES,target_sites=8)
            print('Overture fallback:',json.dumps(alt,indent=2)[:1800])
        except Exception as alt_err:
            print('Overture source unavailable:',type(alt_err).__name__,str(alt_err)[:280])
        status=weak.make_weak_dataset(ROOT,min_evidence=OSM_WEAK_MIN_EVIDENCE,minimum_sites=8)
    print('Auto-matching result:',json.dumps(status,indent=2)[:3500])
    if status['status']=='WEAK_PSEUDO_DATASET_READY_NOT_GROUND_TRUTH':
        exp=ROOT/'weak_osm_experiment'
        report_file=exp/'weak_training_report.json'
        reuse=False
        if report_file.exists() and not RETRAIN_WEAK_MODEL:
            try:
                previous=json.loads(report_file.read_text())
                reuse=(previous.get('status')=='TRAINED_ON_OSM_PSEUDO_LABELS_NOT_VERIFIED'
                       and previous.get('split_sha256')==status['manifest_sha256']
                       and (exp/'weak_best.pt').exists()
                       and weak.sha(exp/'weak_best.pt')==previous.get('weights_sha256'))
            except (ValueError,OSError,TypeError): pass
        print('Step 3/3:', 'reusing existing experimental checkpoint' if reuse else 'training experimental YOLO; this happens only in Notebook 08')
        report=previous if reuse else weak.train_weak_experiment(ROOT,epochs=WEAK_TRAIN_EPOCHS)
        print('Experimental proxy metrics (NOT verified accuracy):',report.get('proxy_metrics'))
        try:
            exported=weak.export_weak_heldout_geojson(ROOT,conf=.35)
            print('Unverified AI map GeoJSON saved:',json.dumps(exported,indent=2))
        except Exception as err:
            print('Weak model trained but map export failed:',type(err).__name__,str(err)[:260])
    else:
        print('No training attempted: insufficient reliably matched independent hospital-building outlines.')
else:
    print('Hands-free OSM AI is OFF by default. Your saved images, annotations, weights and map are untouched.')


### Alternative when the provincial image service is unavailable: an official local GeoTIFF
If the automated downloader is blocked, an independently acquired **licensed high-resolution RGB GeoTIFF** can be placed in Drive. Set `LOCAL_GEOTIFF` to its complete path, supply its provenance, then run this cell. The source must have a real CRS and native ground sampling distance of at most 1 m. This cell is off by default and will not silently overwrite a review manifest.

In [ ]:
# Optional fallback; leave as None if the official image service worked.
LOCAL_GEOTIFF = None  # Example: '/content/drive/MyDrive/your_licensed_ortho.tif'
LOCAL_IMAGERY_SOURCE = 'Geospatial Ontario orthophoto - check collection year'
LOCAL_LICENSE_URL = 'https://data.ontario.ca/dataset/open-ontario-imagery'
if LOCAL_GEOTIFF and not (ROOT/'review_manifest.csv').exists():
    try:
        images = ai.prepare_review_chips_from_geotiff('/content/odhf_facilities.csv',
                    LOCAL_GEOTIFF, ROOT, imagery_license_url=LOCAL_LICENSE_URL,
                    imagery_source_name=LOCAL_IMAGERY_SOURCE, license_confirmed=True)
        print('Prepared local georeferenced images requiring human review:',len(images))
    except Exception as e:
        print('Local GeoTIFF fallback blocked:',type(e).__name__,str(e)[:400])


## 5 — Optional verified roof review (auto-next and OSM draft boxes)

This laboratory shares Notebook 04’s latest reviewer. It can display OSM draft roof proposals (blue dashed, unverified), and moves automatically to the next image after a verified save. Unreviewed drafts never count as training labels.


In [ ]:
# @title Optional manual verification (SKIPPED unless explicitly enabled)
START_MANUAL_REVIEW = False  # No drawing is needed for the weakly supervised experiment
if START_MANUAL_REVIEW:
    if (ROOT/'review_manifest.csv').exists():
        reviewer.launch_labeler(ROOT)
    else:
        print('BLOCKED: no imagery manifest yet. Run the image acquisition cell.')
else:
    print('Manual reviewer skipped. Use the optional OSM weak-supervision cell for no-drawing training.')


## 6 — Convert approved annotations to site-disjoint train/validation/test data

Only run after review. The converter requires explicit human approval, confirmed source and reviewer, and at least three independent positive hospital sites and three independent negative sites. It separates **entire sites**, not adjacent crops, between training, validation and held-out testing. More than three per class is strongly recommended for defensible evaluation.

In [ ]:
try:
    dataset_result=ai.split_verified_dataset(ROOT)
    DATASET_READY=True
    print('SITE-DISJOINT YOLO DATASET:',dataset_result)
except (ValueError,RuntimeError,FileNotFoundError) as e:
    DATASET_READY=False
    print('TRAINING BLOCKED:',str(e))


## 7 — Actually train and evaluate a hospital-building YOLO detector

If the reviewed dataset passes the gate, this cell fine-tunes a real `yolo11n.pt` detector, saves `hospital_best.pt`, and evaluates on never-trained **held-out hospital sites**. It records precision, recall, mAP50, and mAP50–95. When the dataset isn't ready, it does **not** create a model or invent results.

In [ ]:
# Do not retrain an unchanged valid checkpoint on every Colab Run all.
TRAIN_WHEN_READY = False  # Explicitly set True ONLY for the first deliberate training run
ready, why = ai.evaluated_model_ready(ROOT)
if ready:
    print('Reusing verified evaluated model:', why)
elif DATASET_READY and TRAIN_WHEN_READY:
    try:
        import torch
        if not torch.cuda.is_available():
            print('GPU REQUIRED: Runtime > Change runtime type > T4 GPU.')
        else:
            report = ai.train_and_evaluate(ROOT, epochs=40, image_size=640, batch=8)
            print('ACTUAL evaluated metrics:', report['test_metrics'])
    except Exception as e:
        print('TRAINING ERROR:', type(e).__name__, str(e)[:700])
else:
    print('NO TRAINING: human-approved site-separated labels not ready.')


## 8 — Genuine georeferenced predictions for the existing map

Only runs after model training and evaluation. The checkpoint is integrity-checked against its real training report. A **held-out test site's GeoTIFF** is used for demonstration so outputs are not secretly from the training sites. The result is a FeatureCollection of WGS84 points with model scores; its source is an actual trained detector rather than Statistics Canada reference pins.

Import the resulting `.geojson` through the **existing** map's **Map assistant → Advanced: import model prediction GeoJSON** control. You do not need to edit Notebook 04, CSS, filters, or the assistant.

In [ ]:
# Export real detections only on explicit request; reuses trained weights.
EXPORT_PREDICTIONS = False
ready, reason = ai.evaluated_model_ready(ROOT)
print('MODEL STATUS:',reason)
if ready and EXPORT_PREDICTIONS:
    try:
        output=ROOT/'predictions'/'trained_hospital_test_predictions.geojson'
        result=ai.export_all_heldout_predictions(ROOT,output,confidence=.35)
        print('REAL evaluated model output:',result)
        print('Notebook 04 will automatically load this GeoJSON when it next launches.')
    except Exception as exc:
        print('PREDICTION EXPORT BLOCKED:',type(exc).__name__,str(exc)[:700])
else:
    print('Prediction export skipped. Enable EXPORT_PREDICTIONS=True when a model is trained.')


## 9 — Project completion checks

**Required before claiming the hospital AI is complete:** verify the exact orthophoto license/coverage for each chip; manually inspect each hospital roof and negative sample; ensure sites are independent across train/val/test; record true model metrics and false positives/negatives; import model GeoJSON into the unchanged dashboard; publish only code, attribution, appropriate small records and honest outputs to `students/hamza_arab/`.

**Scientific warning:** a hospital's *function* is often invisible from its roof. A strong object detector may find buildings but cannot conclusively distinguish hospitals from schools, hotels or offices without contextual evidence. Treat scores as research screening estimates rather than verified hospital identity.

In [ ]:
from pathlib import Path
print('DATASET:', 'READY' if (ROOT/'hospital_yolo'/'data.yaml').exists() else 'BLOCKED - requires reviewed annotations')
print('MODEL:', 'TRAINED' if (ROOT/'detector'/'training_report.json').exists() else 'NOT TRAINED')
print('PREDICTIONS:', 'EXPORTED' if (ROOT/'predictions'/'trained_hospital_test_predictions.geojson').exists() else 'NOT EXPORTED')
print('Existing map UI: UNMODIFIED. SCREEN_MARGIN_PX in Notebook 04: 65.')


---
[Return to final map (Notebook 01)](https://colab.research.google.com/drive/1YYpGBo-imloznVD9yRzQ4zDmC7EwVOuJ)
